# Phase 11 — Denk-Tiefe vs. Switch: Budget-Forcing-Sweep + Entscheidungsbäume

Die Bestandsdaten (Phase 8b/8c) zeigen: Thinking beseitigt den Switch-Impuls nicht,
es **verlagert** ihn (16 % der Denk-Spuren des Tabellen-Patterns kippen selbst), und
die scheinbare Tiefen-Abhängigkeit ist durch Budget-Starvation + Pattern-Confound
unbestimmbar. Dieses Notebook misst die Übergangstiefe **kontrolliert**:

* dieselben Köder-Prompts über alle Tiefen (kein Pattern-Confound)
* Denk-Budget hart erzwungen: 0/256/512/1024/2048/4096 Tokens, Präfix wird bei
  Budget mit `</think>` geschlossen (kein Starvation-Bias — Antworten immer erzeugt)
* pro (Prompt × Tiefe): mehrere Denk-Präfixe × mehrere Antworten, resumable
* **Entscheidungsbaum am Antwortanfang** pro Tiefenstufe (exakte Wahrscheinlichkeiten)
* Stratifizierung: kippt die Antwort öfter, wenn das Denk-Präfix selbst gekippt war?

A100-Colab, Modell wie Phase 9/10. Ausgaben → Drive (`thinkdepth_*.jsonl`, JSON-Bäume).

In [ ]:
# === Cell 1 — config =========================================================
import os
try:
    from google.colab import userdata
    v=None
    try: v=userdata.get("HF_TOKEN")
    except Exception: v=None
    if v: os.environ.setdefault("HF_TOKEN",v)
except Exception as e:
    print("colab secrets unavailable:", e)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")

MODEL    = os.environ.get("WEIRDSPEC_TARGET_MODEL","Qwen/Qwen3.6-35B-A3B-FP8")
DATA_DIR = "/content/drive/MyDrive/weirdspec"
OUT_PREF = "/content/drive/MyDrive/weirdspec/thinkdepth_prefixes.jsonl"
OUT_ANS  = "/content/drive/MyDrive/weirdspec/thinkdepth_answers.jsonl"
OUT_TREE = "/content/drive/MyDrive/weirdspec/thinkdepth_trees.json"

# the four decoy prompts (full prompt_ids; texts are loaded from weird_transcripts)
PROMPT_IDS = [
  "643fdf5dc86af81b9414db22a33a14ccb94582e2bfdee19816989ef5a280445fa5",  # Tabelle -> JP/KR (17/60 Kipper)
  "6427c18ab919f9d3dac61130206fa5105d6f3cee61fb7282b8d48ee26759f4f7a7",  # TensorFlow -> ZH (7/60)
  "64a66df34dec0e6bca3cd5260271ae1d38ac5380e8c701308acaeeb60226219075",  # Grid -> FR (13 Latein-Kipper)
  "645f57a93e74272bce86e835a201cd660a660ee17f44a06d71daffe3303c61a90f",  # Einkaufsliste -> FR
]
# Tiefen-Gitter v2 (nach Pilot): Aufloesung an den gemessenen Uebergaengen.
# CJK-Kipp stirbt bei <=512 -> feine flache Stufen; FR-Switch faellt erst
# zwischen 512 und 2048 -> mittlere Stufen. 4096 gestrichen (natuerliche
# Denk-Saettigung: Prompts schliessen ihr Denken selbst weit darunter).
DEPTHS_CJK = [0, 64, 128, 256, 512, 1024]
DEPTHS_FR  = [0, 512, 1024, 2048, 3072]
DEPTHS_BY_PID = {p: (DEPTHS_CJK if p.startswith(("643fdf5d","6427c18a")) else DEPTHS_FR)
                 for p in PROMPT_IDS}
DEPTHS = sorted(set(DEPTHS_CJK) | set(DEPTHS_FR))       # Union (fuer die Auswertung)
R_PREFIXES = 4        # Denk-Präfixe pro (Prompt x Tiefe) — gilt jetzt AUCH fuer d=0
                      # (4 leere Präfixe x 8 Antworten = 32er-Baseline statt 8)
ANS_PER_PREFIX = 8    # Antworten pro Präfix  -> 32 Antworten je Stufe
ANS_TOKENS = 24       # reicht zur Onset-Klassifikation
TREE_DEPTH = 6; TREE_PMIN = 0.02; TREE_CHILDREN = 4; TREE_NODES = 60
MOUNT_DRIVE = True
print("model:", MODEL, "| depths:", DEPTHS)

In [ ]:
# === Cell 2 — install + drive + prompts =====================================
import subprocess, sys, os, json, glob
subprocess.run([sys.executable,"-m","pip","install","-q","-U","transformers","accelerate"],check=True)
if MOUNT_DRIVE and not os.path.isdir(DATA_DIR):
    from google.colab import drive; drive.mount("/content/drive")
def read_jsonl(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def find_file(d,*names):
    for nm in names:
        p=os.path.join(d,nm)
        if os.path.isfile(p): return p
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None
wp=find_file(DATA_DIR,"weird_transcripts.jsonl"); assert wp, "weird_transcripts.jsonl fehlt"
PROMPTS={}
for r in read_jsonl(wp):
    pid=r["id"].split("/")[0]
    if pid in PROMPT_IDS and pid not in PROMPTS:
        PROMPTS[pid]=next(t["content"] for t in r["conversations"] if t["role"]=="user")
assert len(PROMPTS)==len(PROMPT_IDS), f"nur {len(PROMPTS)}/{len(PROMPT_IDS)} Prompts gefunden"
for pid,tx in PROMPTS.items(): print(pid[:10],"->",tx[:90].replace("\n"," "))

In [ ]:
# === Cell 3 — model =========================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(MODEL)
model=AutoModelForCausalLM.from_pretrained(MODEL,torch_dtype="auto",device_map="auto")
model.eval()
print("loaded:",model.config.model_type,"| device:",model.device)

In [ ]:
# === Cell 4 — pure logic (mock-testable, no torch) ==========================
import heapq, re
FOREIGN=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
         (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def script_run(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FOREIGN):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
FR=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
EN=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def classify_answer(t):
    """takeover / gloss / latin-switch(fr) / english"""
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FOREIGN)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if script_run(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FR); en=sum(1 for x in w if x in EN)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def tok_script(s):
    for ch in s:
        o=ord(ch)
        if 0x3040<=o<=0x30FF: return "kana"
        if 0x3400<=o<=0x9FFF or 0xF900<=o<=0xFAFF: return "cjk"
        if 0xAC00<=o<=0xD7AF: return "hangul"
        if 0x0400<=o<=0x052F: return "cyrillic"
        if ch.isalpha() and o>=0x0250 and not (0x1E00<=o<=0x1EFF): return "other-nonlatin"
    if any(0x00C0<=ord(c)<=0x017F for c in s): return "latin-acc"
    return "latin"
def expand_tree(next_dist,prefix_ids,depth,p_min,max_children,max_nodes):
    """best-first token tree; next_dist(ids)->(probs desc, token_ids)"""
    root={"tok":None,"p":1.0,"path_p":1.0,"children":[],"_ids":list(prefix_ids),"d":0}
    heap=[(-1.0,0,root)]; n=0; uid=1
    while heap and n<max_nodes:
        _,_,node=heapq.heappop(heap)
        if node["d"]>=depth: continue
        probs,toks=next_dist(node["_ids"])
        for p,t in list(zip(probs,toks))[:max_children]:
            if p<p_min: break
            ch={"tok":int(t),"p":float(p),"path_p":node["path_p"]*float(p),
                "children":[],"_ids":node["_ids"]+[int(t)],"d":node["d"]+1}
            node["children"].append(ch); n+=1
            heapq.heappush(heap,(-ch["path_p"],uid,ch)); uid+=1
            if n>=max_nodes: break
    return root
def decorate(node,decode):
    if node["tok"] is not None:
        s=decode([node["tok"]]); node["text"]=s; node["script"]=tok_script(s)
    node.pop("_ids",None)
    for c in node["children"]: decorate(c,decode)
    return node
def print_tree(node,indent=0,min_p=0.02):
    if node.get("tok") is not None:
        print("  "*indent+"%r  p=%.3f  [%s]"%(node["text"].replace("\n","\\n"),node["p"],node["script"]))
    for c in sorted(node["children"],key=lambda c:-c["p"]):
        if c["p"]>=min_p: print_tree(c,indent+1,min_p)
def foreign_mass(node):
    """probability mass of non-latin branches among the root's children.
    NOTE: blind to the FRENCH switch by construction (latin script) - for the
    FR prompts the answer-level rates are the primary readout, not P_fremd."""
    ch=node["children"]
    return sum(c["p"] for c in ch if c.get("script") in ("kana","cjk","hangul","cyrillic","other-nonlatin"))
def think_prefix(user_text,think_text=None):
    base="<|im_start|>user\n"+user_text+"<|im_end|>\n<|im_start|>assistant\n<think>\n"
    if think_text is None: return base
    return base+think_text+"\n</think>\n\n"
def wilson(k,n,z=1.96):
    import math
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def transition_window(depths,ks,ns):
    """(last depth with rate >= half of depth-0 rate, first depth below) or None"""
    if not ns or ns[0]==0 or ks[0]==0: return None
    base=ks[0]/ns[0]; last_hi=depths[0]; first_lo=None
    for d,k,n in zip(depths[1:],ks[1:],ns[1:]):
        if n==0: continue
        if k/n>=base/2: last_hi=d
        elif first_lo is None: first_lo=d
    return (last_hi,first_lo) if first_lo is not None else None
print("logic ready")

In [ ]:
# === Cell 5 — reasoning prefixes per (prompt x depth), resumable ============
import json, os, torch
END_THINK="</think>"
done=set()
if os.path.exists(OUT_PREF):
    for r in (json.loads(l) for l in open(OUT_PREF) if l.strip()):
        done.add((r["pid"],r["depth"],r["prefix_idx"]))
f=open(OUT_PREF,"a",encoding="utf-8")
@torch.no_grad()
def gen(prefix,max_new,temp=1.0):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=temp,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0,ids.shape[1]:],skip_special_tokens=False)
n_done=0
for pid,user in PROMPTS.items():
    for depth in DEPTHS_BY_PID[pid]:
        for j in range(R_PREFIXES):    # auch d=0: 4 leere Präfixe -> 32 Antworten Baseline
            key=(pid,depth,j)
            if key in done: continue
            if depth==0:
                think=""
            else:
                raw=gen(think_prefix(user),depth)
                think=raw.split(END_THINK)[0]             # Modell schloss selbst frueher
            row=dict(pid=pid,depth=depth,prefix_idx=j,think=think,
                     think_chars=len(think),self_closed=(depth>0 and END_THINK in (raw if depth>0 else "")),
                     think_flipped=script_run(think))
            f.write(json.dumps(row,ensure_ascii=False)+"\n"); f.flush(); n_done+=1
            print("prefix %s d=%-5d #%d chars=%-6d flipped=%s"%(pid[:8],depth,j,len(think),row["think_flipped"]))
f.close(); print("new prefixes:",n_done)

In [ ]:
# === Cell 6 — answers per prefix, resumable =================================
import json, os, torch
prefixes=[json.loads(l) for l in open(OUT_PREF) if l.strip()]
_seen=set(); pfx=[]
for r in prefixes:
    k=(r["pid"],r["depth"],r["prefix_idx"])
    if k not in _seen: pfx.append(r); _seen.add(k)
done=set()
if os.path.exists(OUT_ANS):
    for r in (json.loads(l) for l in open(OUT_ANS) if l.strip()):
        done.add((r["pid"],r["depth"],r["prefix_idx"]))
f=open(OUT_ANS,"a",encoding="utf-8")
@torch.no_grad()
def gen_batch(prefix,n,max_new):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       num_return_sequences=n,pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[1]:],skip_special_tokens=True) for o in out]
for r in pfx:
    key=(r["pid"],r["depth"],r["prefix_idx"])
    if key in done: continue
    full=think_prefix(PROMPTS[r["pid"]],r["think"])
    answers=gen_batch(full,ANS_PER_PREFIX,ANS_TOKENS)
    f.write(json.dumps(dict(pid=r["pid"],depth=r["depth"],prefix_idx=r["prefix_idx"],
        think_flipped=r["think_flipped"],think_chars=r["think_chars"],
        answers=answers,cls=[classify_answer(a) for a in answers]),ensure_ascii=False)+"\n")
    f.flush()
    print("answers %s d=%-5d #%d -> %s"%(r["pid"][:8],r["depth"],r["prefix_idx"],
          [classify_answer(a) for a in answers]))
f.close(); print("answers done")

In [ ]:
# === Cell 7 — decision trees at the answer start, per (prompt x depth) ======
import json, torch, numpy as np
@torch.no_grad()
def next_dist(ids,topn=16):
    t=torch.tensor([ids],device=model.device)
    logits=model(t).logits[0,-1].float()
    p=torch.softmax(logits,-1)
    pr,ix=torch.topk(p,topn)
    return pr.cpu().numpy(),ix.cpu().numpy()
prefixes=[json.loads(l) for l in open(OUT_PREF) if l.strip()]
best={}   # representative prefix per (pid,depth): the first
for r in prefixes:
    best.setdefault((r["pid"],r["depth"]),r)
TREES={}
for (pid,depth),r in sorted(best.items()):
    full=think_prefix(PROMPTS[pid],r["think"])
    ids=tokenizer(full,return_tensors="pt").input_ids[0].tolist()
    tree=expand_tree(next_dist,ids,TREE_DEPTH,TREE_PMIN,TREE_CHILDREN,TREE_NODES)
    decorate(tree,lambda t:tokenizer.decode(t))
    TREES["%s|%d"%(pid,depth)]=dict(pid=pid,depth=depth,think_flipped=r["think_flipped"],
                                    foreign_mass=foreign_mass(tree),tree=tree)
    print("="*90)
    print("TREE %s  depth=%d  (P_fremd am Antwortstart=%.3f, think_flipped=%s)"
          %(pid[:10],depth,foreign_mass(tree),r["think_flipped"]))
    print_tree(tree)
json.dump(TREES,open(OUT_TREE,"w"),ensure_ascii=False)
print("\ntrees ->",OUT_TREE)

In [ ]:
# === Cell 8 — analysis: dose-response, transition window, stratification ====
import json, numpy as np, matplotlib.pyplot as plt, collections
ans=[json.loads(l) for l in open(OUT_ANS) if l.strip()]
_seen=set(); rows=[]
for r in ans:
    k=(r["pid"],r["depth"],r["prefix_idx"])
    if k not in _seen: rows.append(r); _seen.add(k)
SWITCH=("takeover","gloss","latin-switch(fr)")
print("SWITCH-RATE vs erzwungene Denk-Tiefe (Wilson 95%):")
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
pref_by={}
try:
    for r in (json.loads(l) for l in open(OUT_PREF) if l.strip()):
        pref_by.setdefault((r["pid"],r["depth"]),[]).append(r["think_chars"])
except FileNotFoundError: pass
for pid in PROMPT_IDS:
    ds,ps,los,his=[],[],[],[]
    for d in DEPTHS_BY_PID.get(pid,DEPTHS):
        cl=[c for r in rows if r["pid"]==pid and r["depth"]==d for c in r["cls"]]
        if not cl: continue
        k=sum(1 for c in cl if c in SWITCH)
        p,lo,hi=wilson(k,len(cl))
        ds.append(d); ps.append(p); los.append(p-lo); his.append(hi-p)
        mc=pref_by.get((pid,d),[])
        sat=" SATURIERT (natuerl. Denklaenge erreicht)" if (d>0 and mc and
             sum(mc)/len(mc)<3.5*d) else ""
        print("  %s d=%-5d n=%-3d rate=%.1f%% [%.1f,%.1f]  %s%s"%(pid[:8],d,len(cl),100*p,
              100*(p-los[-1]),100*(p+his[-1]),dict(collections.Counter(cl)),sat))
    ax[0].errorbar([d+1 for d in ds],[100*p for p in ps],yerr=[[100*l for l in los],[100*h for h in his]],
                   marker="o",capsize=3,label=pid[:8])
    tw=transition_window(ds,[sum(1 for r in rows if r["pid"]==pid and r["depth"]==d
        for c in r["cls"] if c in SWITCH) for d in ds],
        [sum(len(r["cls"]) for r in rows if r["pid"]==pid and r["depth"]==d) for d in ds])
    # Hinweis: P_fremd der Baeume ist fuer FR-Prompts konstruktionsbedingt blind
    print("  -> Uebergangsfenster %s: %s"%(pid[:8],tw))
ax[0].set_xscale("symlog"); ax[0].set_xlabel("erzwungene Denk-Tiefe (Tokens, +1)")
ax[0].set_ylabel("Switch-Rate der Antwort (%)"); ax[0].legend(fontsize=8,frameon=False)
ax[0].set_title("Dosis-Wirkung: Denk-Tiefe vs. Switch")
# stratification: flipped vs clean thinking prefixes (depth>0 pooled)
lab=[]; vals=[]
for fl in (False,True):
    cl=[c for r in rows if r["depth"]>0 and r["think_flipped"]==fl for c in r["cls"]]
    k=sum(1 for c in cl if c in SWITCH); p,lo,hi=wilson(k,len(cl))
    lab.append("Denken sauber\n(n=%d)"%len(cl) if not fl else "Denken gekippt\n(n=%d)"%len(cl))
    vals.append((100*p,100*(p-lo),100*(hi-p)))
    print("think_flipped=%s: rate=%.1f%% [%.1f,%.1f] n=%d"%(fl,100*p,100*lo,100*hi,len(cl)))
ax[1].bar(range(2),[v[0] for v in vals],yerr=[[v[1] for v in vals],[v[2] for v in vals]],
          capsize=4,color=["#2563EB","#DC2626"],alpha=0.85)
ax[1].set_xticks(range(2)); ax[1].set_xticklabels(lab,fontsize=9)
ax[1].set_ylabel("Switch-Rate der Antwort (%)")
ax[1].set_title("Verlagerungs-These: kippt die Antwort oefter,\nwenn das Denken selbst gekippt war?")
# tree summary: P_foreign at answer start vs depth
try:
    TREES=json.load(open(OUT_TREE))
    print("\nP(fremdes Token) am Antwortstart (aus den Baeumen):")
    for key,t in sorted(TREES.items()):
        print("  %s d=%-5d P_fremd=%.3f"%(t["pid"][:8],t["depth"],t["foreign_mass"]))
except FileNotFoundError:
    print("(Baeume noch nicht erzeugt - Cell 7)")
plt.tight_layout(); plt.show()

## Cell 9 — Prompt-Perplexität: Wie künstlich sind die Köder wirklich?

Perplexität jedes Language-Switching-Prompts unter dem Subjektmodell, doppelt
verankert (16 natürliche Nutzer-Prompts inkl. drei Köder-Zwillingen + Katalog-
Prompts der anderen Behaviors), kombiniert mit Naturalness-Elo, match_rate und
Köder-Index in einer Tabelle. Läuft in der Phase-11-Session (Modell geladen).

In [ ]:
# === Cell 9 — Prompt-Perplexität: Köder vs. natürliche Prompts ==============
# Misst die Perplexität jedes Language-Switching-Prompts unter dem SUBJEKT-
# Modell (als User-Turn im Chat-Template) und verankert sie doppelt:
#   * 16 handgeschriebene natürliche Nutzer-Prompts (darunter drei "Zwillinge"
#     der Köder: echte TensorFlow-Frage, echte Einkaufsliste, echtes 10x10-Grid)
#   * die Prompts der übrigen Behaviors des Katalogs (bis 120, als Verteilung)
# Ausgabe: EINE kombinierte Tabelle mit Naturalness-Elo, P_nat, match_rate,
# Köder-Index und PPL-Ankern. Braucht model+tokenizer aus Cell 3 + Drive.
import json, math, os, glob, statistics, collections, torch, numpy as np
DATA_DIR=globals().get("DATA_DIR","/content/drive/MyDrive/weirdspec")
def _rj(p):
    rows=[]
    with open(p,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def _ff(d,*names):
    for nm in names:
        q=os.path.join(d,nm)
        if os.path.isfile(q): return q
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None
NATURAL=[
 "Can you help me write a polite email to my landlord about a broken heater? It's been three days and the apartment is getting cold.",
 "What's the difference between a Roth IRA and a traditional IRA? I'm 28 and just started my first full-time job.",
 "My pasta sauce always turns out too acidic. What can I do to balance it?",
 "Explain how HTTPS works in simple terms.",
 "I have a 10x10 grid in Excel and want to highlight every cell above the column average. How do I do that with conditional formatting?",
 "Write a short bedtime story about a dragon who is afraid of the dark.",
 "What should I see in Lisbon in three days? We like food and museums, not so much nightlife.",
 "Can you review this sentence for grammar: 'Neither of the managers were available for comments.'",
 "How do I center a div horizontally and vertically in CSS?",
 "Summarize the plot of Hamlet in five sentences.",
 "My dog keeps pulling on the leash. Any training tips?",
 "What is TensorFlow and what is it used for?",
 "Please make me a grocery list for a week of vegetarian dinners for two people.",
 "Convert this table of monthly expenses into a budget summary: rent 1200, food 400, transport 90, subscriptions 35.",
 "Why is the sky blue during the day but red at sunset?",
 "Draft a LinkedIn post announcing that I'm starting a new role as a data analyst.",
]
HDR_IDS=tokenizer("<|im_start|>user\n").input_ids
@torch.no_grad()
def prompt_ppl(text):
    full="<|im_start|>user\n"+text+"<|im_end|>"
    ids=tokenizer(full,return_tensors="pt",truncation=True,max_length=768).input_ids.to(model.device)
    lp=model(ids).logits[0,:-1].float().log_softmax(-1)
    tgt=ids[0,1:]
    lps=lp[torch.arange(len(tgt)),tgt][len(HDR_IDS)-1:-1]   # nur Prompt-Inhalt, ohne im_end
    return float(torch.exp(-lps.mean())) if lps.numel() else float("nan")

mp=_ff(DATA_DIR,"weird_meta.jsonl"); wp=_ff(DATA_DIR,"weird_transcripts.jsonl")
meta=_rj(mp); id2m={r["transcript_id"]:r for r in meta}
LS="language-switching-english"
# repräsentativer Prompt je LS-Pattern = häufigste prompt_id
byp=collections.defaultdict(collections.Counter)
elo={}; mrate={}
for r in meta:
    if r["behavior_id"]==LS:
        byp[r["pattern_id"]][r["prompt_id"]]+=1
        elo[r["pattern_id"]]=r.get("elo_prompt_naturalness"); mrate[r["pattern_id"]]=r.get("match_rate")
rep={pat:c.most_common(1)[0][0] for pat,c in byp.items()}
need={pid for pid in rep.values()}
other_ids={}   # prompt_id -> (behavior) fuer Katalog-Referenz
for r in meta:
    if r["behavior_id"]!=LS and r["prompt_id"] not in other_ids:
        other_ids[r["prompt_id"]]=r["behavior_id"]
texts={}; other_texts={}
for r in _rj(wp):
    pid=r["id"].split("/")[0]
    if pid in need and pid not in texts:
        texts[pid]=next(t["content"] for t in r["conversations"] if t["role"]=="user")
    elif pid in other_ids and pid not in other_texts and len(other_texts)<120:
        other_texts[pid]=next(t["content"] for t in r["conversations"] if t["role"]=="user")
print("score: %d LS-Prompts, %d natuerliche Anker, %d Katalog-Prompts"%(len(texts),len(NATURAL),len(other_texts)))
nat_ppl=[prompt_ppl(t) for t in NATURAL]
nat_med=statistics.median(nat_ppl)
cat_ppl=sorted(prompt_ppl(t) for t in other_texts.values())
def cat_pct(x): return 100*sum(1 for v in cat_ppl if v<x)/len(cat_ppl)
allelo=sorted(v for v in (r.get("elo_prompt_naturalness") for r in meta) if v is not None)
med_elo=statistics.median(allelo)
def pnat(e): return 1/(1+10**((med_elo-e)/400)) if e is not None else float("nan")

print("\nnatuerliche Anker: PPL median %.1f (min %.1f, max %.1f)"%(nat_med,min(nat_ppl),max(nat_ppl)))
print("Katalog-Referenz (andere Behaviors): PPL median %.1f\n"%statistics.median(cat_ppl))
rowsout=[]
for pat,pid in rep.items():
    if pid not in texts: continue
    ppl=prompt_ppl(texts[pid]); e=elo[pat]; m=mrate[pat]
    rowsout.append((pat.split(LS+"/")[1],pid[:8],e,pnat(e),m,m*pnat(e),ppl,ppl/nat_med,cat_pct(ppl)))
print("%-26s %-9s %5s %7s %6s %8s %8s %10s %9s"%("pattern","prompt","elo","P_nat","match","K-Index","PPL","x nat.Med","Kat-Pzt"))
for r in sorted(rowsout,key=lambda r:-r[5]):
    print("%-26s %-9s %5.0f  %6.3f  %5.3f  %7.4f  %7.1f  %8.1fx  %7.0f%%"%r)
print("\nZWILLINGS-VERGLEICH (Koeder vs. natuerliche Version desselben Anliegens):")
tw={"TensorFlow":("6427c18a",NATURAL[11]),"Einkaufsliste":("645f57a9",NATURAL[12]),"10x10-Grid":("64a66df34",NATURAL[4])}
for name,(pfx,nat) in tw.items():
    k=[t for pid,t in texts.items() if pid.startswith(pfx[:8])]
    if k:
        print("  %-14s Koeder-PPL %7.1f  vs. natuerlich %6.1f  -> Faktor %.1fx"
              %(name,prompt_ppl(k[0]),prompt_ppl(nat),prompt_ppl(k[0])/prompt_ppl(nat)))
PPL_RESULTS=dict(nat_median=nat_med,rows=rowsout)


## Cell 10 — Nachauswertung (offline, CPU)

Cluster-bewusste Dosis-Tabelle (Bootstrap über Denk-Präfixe statt Antworten),
korrigierte Baum-Metrik P_erstFremd (Masse des ersten Fremd-Eintritts entlang
aller Pfade) mit Konsistenz-Check gegen die empirischen Schrift-Raten, und die
Anatomie des gekippten Gedankengangs. Braucht nur die drei Drive-Dateien.

In [ ]:
# === Cell 10 — Nachauswertung (offline, CPU): Cluster, Baum-Fix, der Flip ===
# Repariert die drei Schwachstellen des Hauptlaufs, ohne neue GPU-Zeit:
#  (a) CLUSTER-BEWUSSTE Dosis-Tabelle: Antworten haengen an 4 Denk-Praefixen
#      je Stufe -> Rate je Praefix + Bootstrap UEBER Praefixe (ehrliche CIs);
#      die Roh-Zaehlung je Praefix zeigt sofort, ob FR-"Buckel" echt sind
#      oder an einem einzelnen gefaehrlichen Gedankengang haengen.
#  (b) P_erstFremd: korrigierte Baum-Metrik = Wahrscheinlichkeitsmasse des
#      ERSTEN Fremd-Eintritts entlang aller Pfade (path_p der obersten
#      fremden Knoten) statt nur Wurzelkinder; Konsistenz-Check gegen die
#      empirischen Schrift-Raten (CJK-Prompts).
#  (c) Der gekippte Gedankengang: wo im Denk-Text der Flip beginnt, was
#      danach kam, und wie seine 8 Antworten klassifiziert wurden.
import os, json, math, collections, numpy as np, matplotlib.pyplot as plt
OUT_PREF=globals().get("OUT_PREF","/content/drive/MyDrive/weirdspec/thinkdepth_prefixes.jsonl")
OUT_ANS =globals().get("OUT_ANS","/content/drive/MyDrive/weirdspec/thinkdepth_answers.jsonl")
OUT_TREE=globals().get("OUT_TREE","/content/drive/MyDrive/weirdspec/thinkdepth_trees.json")
if not os.path.exists(OUT_ANS):
    from google.colab import drive; drive.mount("/content/drive")
def _rj(p):
    rows=[]
    with open(p,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
SWITCH=("takeover","gloss","latin-switch(fr)"); SCRIPT_SW=("takeover","gloss")
_s=set(); ANS=[]
for r in _rj(OUT_ANS):
    k=(r["pid"],r["depth"],r["prefix_idx"])
    if k not in _s: ANS.append(r); _s.add(k)
_s=set(); PREF=[]
for r in _rj(OUT_PREF):
    k=(r["pid"],r["depth"],r["prefix_idx"])
    if k not in _s: PREF.append(r); _s.add(k)
by=collections.defaultdict(list)
for r in ANS: by[(r["pid"],r["depth"])].append(r)
pids=sorted(set(p for p,_ in by))

# ---------------- (a) cluster-aware dose table ------------------------------
rng=np.random.default_rng(0)
print("CLUSTER-BEWUSSTE DOSIS-TABELLE (Rate je Praefix | Bootstrap ueber Praefixe):")
CLUSTER={}
fig,ax=plt.subplots(1,2,figsize=(12.5,4.4))
for pid in pids:
    ds,ms,lo_,hi_=[],[],[],[]
    print("  "+pid[:8]+":")
    for (p2,d),rows in sorted(by.items(),key=lambda kv:kv[0][1]):
        if p2!=pid: continue
        fr=np.array([sum(1 for c in r["cls"] if c in SWITCH)/len(r["cls"]) for r in rows])
        boots=[float(np.mean(rng.choice(fr,len(fr)))) for _ in range(2000)]
        lo,hi=np.percentile(boots,[2.5,97.5])
        per=" ".join("%d/%d"%(sum(1 for c in r["cls"] if c in SWITCH),len(r["cls"])) for r in rows)
        print("    d=%-5d mean=%5.1f%%  [%4.1f,%4.1f]  praefixe: %s"%(d,100*fr.mean(),100*lo,100*hi,per))
        CLUSTER[(pid,d)]=(float(fr.mean()),float(lo),float(hi))
        ds.append(d); ms.append(100*fr.mean()); lo_.append(100*(fr.mean()-lo)); hi_.append(100*(hi-fr.mean()))
    ax[0].errorbar([d+1 for d in ds],ms,yerr=[lo_,hi_],marker="o",capsize=3,label=pid[:8])
ax[0].set_xscale("symlog"); ax[0].set_xlabel("erzwungene Denk-Tiefe (Tokens, +1)")
ax[0].set_ylabel("Switch-Rate (%)"); ax[0].legend(fontsize=8,frameon=False)
ax[0].set_title("Dosis-Wirkung, CIs via Bootstrap ueber Praefixe\n(ehrlich gegen Antwort-Clustering)")

# ---------------- (b) corrected tree metric ---------------------------------
FOR=("kana","cjk","hangul","cyrillic","other-nonlatin")
def first_foreign_mass(node,anc=False):
    isf=node.get("script") in FOR
    if isf and not anc: return float(node.get("path_p",0.0))
    return sum(first_foreign_mass(c,anc or isf) for c in node.get("children",[]))
sc=[]
if os.path.exists(OUT_TREE):
    TREES=json.load(open(OUT_TREE))
    print("\nP_erstFremd (korrigierte Baum-Metrik) vs. empirische SCHRIFT-Rate:")
    for key,t in sorted(TREES.items(),key=lambda kv:(kv[1]["pid"],kv[1]["depth"])):
        pf=first_foreign_mass(t["tree"])
        rows=by.get((t["pid"],t["depth"]),[])
        emp=(sum(1 for r in rows for c in r["cls"] if c in SCRIPT_SW)
             /sum(len(r["cls"]) for r in rows)) if rows else float("nan")
        print("  %s d=%-5d P_erstFremd=%.3f  empirisch=%s"%(t["pid"][:8],t["depth"],pf,
              ("%.3f"%emp) if emp==emp else "-"))
        if emp==emp: sc.append((t["pid"],pf,emp))
    cj=[(pf,emp) for pid,pf,emp in sc if pid[:8] in ("643fdf5d","6427c18a")]
    if len(cj)>=4:
        x=np.array([a for a,_ in cj]); y=np.array([b for _,b in cj])
        pear=float(np.corrcoef(x,y)[0,1])
        rs=float(np.corrcoef(np.argsort(np.argsort(x)),np.argsort(np.argsort(y)))[0,1])
        print("  Konsistenz (CJK-Prompts): Pearson r=%.2f | Spearman rho=%.2f (n=%d)"%(pear,rs,len(cj)))
        ax[1].scatter(x,y,color="#2563EB")
        m=max(0.01,x.max(),y.max())
        ax[1].plot([0,m],[0,m],ls=":",color="black",lw=.8)
        ax[1].set_xlabel("P_erstFremd (Baum, 1 Praefix)"); ax[1].set_ylabel("empirische Schrift-Rate (32 Antworten)")
        ax[1].set_title("Baum vs. Empirie (CJK)  r=%.2f"%pear)
    fr_note=[(pid,pf) for pid,pf,_ in sc if pid[:8] in ("64a66df3","645f57a9") and pf>0.01]
    print("  (FR-Prompts: P_erstFremd ist fuer den Franzoesisch-Ast konstruktionsbedingt blind)")

# ---------------- (c) the flipped reasoning ---------------------------------
FRANGE=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
        (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def flip_pos(t,run=3):
    c=0; start=None
    for i,ch in enumerate(t):
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRANGE):
            if c==0: start=i
            c+=1
            if c>=run: return start
        elif ch.isalpha(): c=0
    return None
fl=[r for r in PREF if r.get("think_flipped")]
print("\nGEKIPPTE DENK-PRAEFIXE: %d von %d"%(len(fl),len(PREF)))
for r in fl:
    t=r["think"]; pos=flip_pos(t)
    arow=[x for x in by.get((r["pid"],r["depth"]),[]) if x["prefix_idx"]==r["prefix_idx"]]
    print("  %s d=%d #%d  Flip ab Zeichen %s/%d (%.0f%% der Passage)"
          %(r["pid"][:8],r["depth"],r["prefix_idx"],pos,len(t),100*(pos or 0)/max(len(t),1)))
    if arow: print("    Antworten danach:",arow[0]["cls"])
    if pos is not None:
        print("    Kontext: …%s…"%t[max(0,pos-100):pos+200].replace("\n"," "))
plt.tight_layout(); plt.show()
POST_RESULTS=dict(cluster={("%s|%d"%k):v for k,v in CLUSTER.items()},
                  n_flipped=len(fl))


## Cell 11 — Memorisierungs-Basin-Test

Reicht der nackte japanische Tabellenkopf (Roh-Text, kein Prompt, keine
Aufgabe), um das kanonische Cloud-Speicher-Vergleichs-Set (Google Drive /
Dropbox / OneDrive, 15/2/5 GB) hervorzuholen? Vier Bedingungen (JP/KR/EN-
Header + neutraler JP-Header als Kontrolle), 16 Fortsetzungen je Bedingung.

In [ ]:
# === Cell 11 — Memorisierungs-Basin-Test ====================================
# Hypothese (Daten-Rueckgriff ohne Wiederholungs-Trigger): Der JP-Tabellen-
# Kipp ist teilweise FORMAT-RETRIEVAL aus einem dichten Trainings-Modus
# (japanische Cloud-Speicher-Vergleichstabellen mit kanonischen Fakten
# 15/2/5 GB). Test: Das Modell bekommt NUR einen nackten Tabellenkopf als
# ROH-TEXT (kein Chat-Template, kein User-Prompt, keine Aufgabe) und wir
# messen, ob es das kanonische Vergleichs-Set von selbst hervorholt.
#   A  JP-Header Speicher   | サービス名 | ストレージ制限 |
#   B  KR-Header Speicher   | 서비스명 | 저장 용량 |
#   C  EN-Header Speicher   | Service Name | Storage Limit |   (Fakten-Basis)
#   D  JP-Header NEUTRAL    | 国名 | 人口 |     (generischer JP-Tabellen-Modus)
# Braucht nur model+tokenizer (Cells 1-3), keine Drive-Daten. ~2-4 min.
import re, math, torch, collections
N_SAMPLES=16; MAX_NEW=96
CONDS={
 "A JP-Speicher": "| サービス名 | ストレージ制限 |\n|---|---|\n",
 "B KR-Speicher": "| 서비스명 | 저장 용량 |\n|---|---|\n",
 "C EN-Speicher": "| Service Name | Storage Limit |\n|---|---|\n",
 "D JP-neutral":  "| 国名 | 人口 |\n|---|---|\n",
}
TRIO=[re.compile(p,re.I) for p in
      (r"google\s*drive|グーグル\s*・?\s*ドライブ|谷歌云盘|구글\s*드라이브",
       r"dropbox|ドロップボックス|드롭박스",
       r"onedrive|ワンドライブ|원드라이브")]
CANON=[re.compile(p,re.I) for p in
       (r"(?<![\d.,])15\s*(gb|ギガ|기가)",r"(?<![\d.,])2\s*(gb|ギガ|기가)",
        r"(?<![\d.,])5\s*(gb|ギガ|기가)")]
def score(t):
    """(#Trio-Dienste erwaehnt, #kanonische Zahlen, Fremdschrift-Anteil)"""
    trio=sum(1 for rx in TRIO if rx.search(t))
    canon=sum(1 for rx in CANON if rx.search(t))
    al=[ch for ch in t if ch.isalpha()]
    fo=sum(1 for ch in al if ord(ch)>=0x250)
    return trio,canon,(fo/len(al) if al else 0.0)
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
@torch.no_grad()
def cont(prefix,n):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                       num_return_sequences=n,pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[1]:],skip_special_tokens=True) for o in out]
RES={}
for name,pre in CONDS.items():
    outs=cont(pre,N_SAMPLES)
    sc=[score(t) for t in outs]
    k_trio=sum(1 for a,b,c in sc if a==3); k_can=sum(1 for a,b,c in sc if b>=2)
    RES[name]=dict(outs=outs,k_trio=k_trio,k_can=k_can)
    p1=wilson(k_trio,N_SAMPLES); p2=wilson(k_can,N_SAMPLES)
    fs=sum(c for _,_,c in sc)/len(sc)
    print("%-14s Trio komplett: %2d/%d (%3.0f%% [%3.0f,%3.0f])  >=2 kanon. Zahlen: %2d/%d (%3.0f%% [%3.0f,%3.0f])  Fremdanteil %3.0f%%"
          %(name,k_trio,N_SAMPLES,100*p1[0],100*p1[1],100*p1[2],
            k_can,N_SAMPLES,100*p2[0],100*p2[1],100*p2[2],100*fs))
print("\nBEISPIELE (erste Fortsetzung je Bedingung):")
for name in CONDS:
    print("--- %s ---"%name); print(RES[name]["outs"][0][:400].strip()); print()
a=RES["A JP-Speicher"]; d=RES["D JP-neutral"]
print("VERDIKT:")
if a["k_trio"]>=N_SAMPLES*0.5 and a["k_trio"]>=3*max(1,d["k_trio"]):
    print("  BASIN: Der nackte JP-Speicher-Header genuegt, um das kanonische")
    print("  Vergleichs-Set hervorzuholen - Format-Retrieval aus einem dichten")
    print("  Trainings-Modus ist ein realer Bestandteil des Kipps.")
elif a["k_trio"]<=N_SAMPLES*0.2:
    print("  KEIN BASIN: Der Header allein holt das kanonische Set kaum hervor -")
    print("  der Kipp braucht die Aufgabenstellung; reines Format-Retrieval traegt wenig.")
else:
    print("  TEILBEFUND: maessige Retrieval-Rate - Basin existiert, traegt aber nicht allein.")
print("  (C zeigt, ob die FAKTEN sprachunabhaengig allgegenwaertig sind;")
print("   D kontrolliert den generischen JP-Tabellen-Modus.)")
BASIN_RESULTS={k:dict(k_trio=v["k_trio"],k_can=v["k_can"]) for k,v in RES.items()}


## Cell 12 — Prefill-only-Ablation: Wo entsteht die Kipp-Disposition?

Die 317 Takeover-Experten werden nur im Prefill (Prompteintritt) bzw. nur in
der Generierung maskiert — vier Arme (none/prefill/gen/full), JP-Tabelle als
Kernfrage plus FR-Grid als Dissoziations-Kontrolle. Phasen-Erkennung über die
Zeilenzahl der Router-Aufrufe (Prefill: batch×prompt_len, Gen-Schritt: batch).
Laufzeit-Verifikation im Hook (banned nie im top-k maskierter Aufrufe).

In [ ]:
# === Cell 12 — PREFILL-ONLY-ABLATION: wo entsteht die Kipp-Disposition? =====
# Kausaltest der Prompteintritts-Frage: Die 317 Takeover-Experten werden
# NUR waehrend der Prompt-Verarbeitung stummgeschaltet (Prefill), waehrend
# der Generierung aber freigegeben - und umgekehrt. Vier Arme:
#   none     keine Maskierung                (Baseline, erwartet ~40% Tabelle)
#   prefill  Maske NUR im Prefill            <- die eigentliche Frage
#   gen      Maske NUR waehrend Generierung  (Phase-10-Logik, erwartet ~0)
#   full     Maske immer                     (Replikation Phase 10, ~0)
# Phasen-Erkennung: mit KV-Cache sieht der Router im Prefill batch*prompt_len
# Token-Zeilen, bei jedem Generierungs-Schritt genau batch Zeilen.
# Grenz-Feinheit (ehrlich): Der Prefill-Forward erzeugt auch die Logits des
# ERSTEN gesampelten Tokens - "prefill" umfasst also die Wahl von Antwort-
# Token 0; der Kipp selbst (Token 1, z.B. サービス) faellt in die Gen-Phase.
# Prompts: JP-Tabelle (317 = Schrift-Modul, Kernfrage) + FR-Grid (Kontrolle:
# ist der Latein-Switch ueberhaupt 317-abhaengig? -> doppelte Dissoziation).
# Braucht model+tokenizer (Cells 1-3), PROMPTS (Cell 2), banned_experts.json.
import json, os, re, math, torch, collections
BANNED_JSON=globals().get("BANNED_JSON","/content/drive/MyDrive/weirdspec/banned_experts.json")
N_ANS=32; MAX_NEW=24; TOP_K=8
PIDS=[p for p in PROMPT_IDS if p.startswith(("643fdf5d","64a66df3"))]
banned={int(l):v for l,v in json.load(open(BANNED_JSON)).items()}
E=getattr(model.config,"num_experts",None) or getattr(model.config,"n_routed_experts",None) or 256
H=model.config.hidden_size
gates={}
for name,mod in model.named_modules():
    w=getattr(mod,"weight",None)
    if w is None or tuple(w.shape) not in [(E,H),(H,E)]: continue
    low=name.lower()
    if any(b in low for b in ("shared","attn","proj")): continue
    if not ("gate" in low or "router" in low or "moe" in low): continue
    m=re.search(r"layers\.(\d+)\.",name)
    if m: gates[int(m.group(1))]=mod
assert gates, "keine Router gefunden"
masks={l:torch.zeros(E,dtype=torch.bool) for l in gates}
for l,ids in banned.items():
    if l in masks:
        for e in ids: masks[l][e]=True
print("gates: %d Layer | banned-Layer mit Eintraegen: %d"%(len(gates),sum(1 for l in gates if masks[l].any())))

STATE={"mode":"none","n_ret":1,"masked":0,"free":0,"verified":False}
def should_mask(rows,mode,n_ret):
    """prefill-Aufrufe haben rows>n_ret (batch*prompt_len), Gen-Schritte rows==n_ret"""
    if mode=="none": return False
    if mode=="full": return True
    is_prefill=rows>n_ret
    return is_prefill if mode=="prefill" else (not is_prefill)
def mk_hook(l):
    mvec=masks[l]
    def hook(mod,inp,out):
        t=out[0] if isinstance(out,tuple) else out
        if t.shape[-1]!=E: return out
        rows=int(t.reshape(-1,E).shape[0])
        if not should_mask(rows,STATE["mode"],STATE["n_ret"]):
            STATE["free"]+=1; return out
        STATE["masked"]+=1
        ml=t.reshape(-1,E).masked_fill(mvec.to(t.device),torch.finfo(t.dtype).min/2).reshape(t.shape)
        if not STATE["verified"] and mvec.any():
            _,ti=torch.topk(ml.reshape(-1,E),TOP_K,dim=-1)
            assert not mvec.to(ti.device)[ti.reshape(-1)].any(), "VERIFY FAIL: banned in top-k"
            STATE["verified"]=True
        if not isinstance(out,tuple): return ml
        new=[ml]
        for x in out[1:]:
            if torch.is_tensor(x) and x.shape[-1]==TOP_K:
                tv,ti=torch.topk(ml.reshape(-1,E),TOP_K,dim=-1)
                if x.dtype in (torch.int32,torch.int64): new.append(ti.reshape(x.shape))
                else: new.append(torch.softmax(tv,-1).to(x.dtype).reshape(x.shape))
            else: new.append(x)
        return tuple(new)
    return hook
handles=[gates[l].register_forward_hook(mk_hook(l)) for l in gates]

FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=o2<=b for a,b in FRW for o2 in [ord(ch)]):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def think_prefix(u,th=""):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def twoprop(k1,n1,k2,n2):
    """z-Test zweier Anteile -> p-Wert (zweiseitig)"""
    p=(k1+k2)/(n1+n2)
    se=math.sqrt(p*(1-p)*(1/n1+1/n2))
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
@torch.no_grad()
def gen_batch(prefix,n,max_new):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       num_return_sequences=n,pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[1]:],skip_special_tokens=True) for o in out]

ARMS=("none","prefill","gen","full"); RES={}
SW=("takeover","gloss","latin-switch(fr)")
for pid in PIDS:
    prefix=think_prefix(PROMPTS[pid],"")
    for arm in ARMS:
        STATE.update(mode=arm,n_ret=N_ANS,masked=0,free=0,verified=False)
        cls=[classify_answer(a) for a in gen_batch(prefix,N_ANS,MAX_NEW)]
        k=sum(1 for c in cls if c in SW)
        RES[(pid,arm)]=(k,N_ANS,dict(collections.Counter(cls)))
        p,lo,hi=wilson(k,N_ANS)
        print("%s %-8s rate=%5.1f%% [%4.1f,%4.1f]  masked_calls=%-5d free_calls=%-5d %s"
              %(pid[:8],arm,100*p,100*lo,100*hi,STATE["masked"],STATE["free"],RES[(pid,arm)][2]))
for h in handles: h.remove()
print("\nVERDIKT:")
for pid in PIDS:
    kb,nb,_=RES[(pid,"none")]; kp,np_,_=RES[(pid,"prefill")]
    kg,ng,_=RES[(pid,"gen")]; kf,nf,_=RES[(pid,"full")]
    p_pre=twoprop(kb,nb,kp,np_); p_gen=twoprop(kb,nb,kg,ng)
    print("  %s baseline %d/%d | prefill-only %d/%d (p=%.3f) | gen-only %d/%d (p=%.3f) | full %d/%d"
          %(pid[:8],kb,nb,kp,np_,p_pre,kg,ng,p_gen,kf,nf))
    if pid.startswith("643fdf5d"):
        if kf>nb*0.1: print("    !! full-Arm nicht ~0 - Hooks pruefen, Aussagen unten unter Vorbehalt")
        if p_pre<0.05 and kp<kb: print("    -> DISPOSITION IM PROMPTEINTRITT: Prefill-Maskierung allein drueckt den Kipp.")
        elif p_pre>=0.05 and p_gen<0.05: print("    -> ENTSCHEIDUNG BEI DER GENERIERUNG: Prefill-Aktivitaet der 317 ist entbehrlich;")
        else: print("    -> unklar/teilweise - Zahlen oben ansehen.")
    else:
        allp=[twoprop(kb,nb,*RES[(pid,a)][:2]) for a in ("prefill","gen","full")]
        if min(allp)>=0.05: print("    -> FR unabhaengig von den 317 (keine Maskierung wirkt) - doppelte Dissoziation der Mechanismen.")
        else: print("    -> FR reagiert auf 317-Maskierung - Latein-Switch teilt Maschinerie mit dem Schrift-Modul.")
PREFILL_RESULTS={("%s|%s"%k):v[:2] for k,v in RES.items()}


## Cell 13 — Fülltext-Kontrolle: Verdünnung (TTT) vs. Inhalt

Injiziert neutralen englischen Fülltext exakt gleicher Token-Länge statt echten
Denkens in den Think-Block (gleiches Gitter wie der echte Lauf) und vergleicht
beide Dosis-Kurven. Deckungsgleich → Fast-Weight-Verdünnung (TTT-Lesart);
Fülltext wirkungslos → Deliberations-Inhalt (Vorentwurf). Resumable, ~10–15 min.

In [ ]:
# === Cell 13 — FUELLTEXT-KONTROLLE: Verduennung (TTT) vs. Inhalt ============
# Entscheidet zwischen zwei Erklaerungen der Denk-Unterdrueckung:
#   VERDUENNUNG (TTT/Fast-Weights): JEDES englische Material im Think-Block
#     verschiebt den Kontextzustand -> auch themenfremder Fuelltext drueckt
#     die Kipp-Rate, Kurve ~ deckungsgleich mit echtem Denken.
#   INHALT (Vorentwurf/Deliberation): nur aufgabenbezogenes Denken hilft ->
#     Fuelltext-Kurve bleibt nahe der Baseline.
# Design: statt echten Denkens wird NEUTRALER englischer Fuelltext exakt
# gleicher Token-Laenge in den Think-Block injiziert - gleiches Gitter wie
# der echte Lauf, 4 Text-Varianten x 8 Antworten je Stufe, resumable.
# Ehrliche Grenze: injizierter Text ist off-policy (das Modell hat ihn nie
# selbst erzeugt) - der Vergleich gilt der SPRACHSTATISTIK, nicht der Frage,
# ob das Modell "so haette denken koennen".
import json, os, math, re, torch, collections, numpy as np, matplotlib.pyplot as plt
OUT_FILL="/content/drive/MyDrive/weirdspec/thinkfiller_answers.jsonl"
OUT_ANS =globals().get("OUT_ANS","/content/drive/MyDrive/weirdspec/thinkdepth_answers.jsonl")
N_VARIANTS=4; ANS_PER=8; MAX_NEW=24
PIDS=[p for p in PROMPT_IDS if p.startswith(("643fdf5d","64a66df3"))]
GRID={p:(DEPTHS_BY_PID.get(p,DEPTHS) if "DEPTHS_BY_PID" in globals() else [0,64,128,256,512,1024])
      for p in PIDS}
SENT_BANK=[
 "The old hiking trail winds slowly through the pine forest before it reaches the ridge.",
 "In early spring the meltwater streams are loud enough to be heard from the village below.",
 "A pair of buzzards circled above the valley for most of the warm afternoon.",
 "The lighthouse keeper kept a small garden of potatoes and kale behind the storage shed.",
 "During the long winter months the harbor freezes over and the ferries stop running.",
 "She repaired the wooden fence on Saturday and painted it a pale shade of grey.",
 "The museum's east wing holds a collection of nineteenth century farming tools.",
 "Fresh bread from the corner bakery sells out most mornings before nine o'clock.",
 "The night train to the coast passes the old mill exactly at a quarter past two.",
 "His grandfather taught him to read the weather from the shape of the evening clouds.",
 "The chess club meets every second Thursday in the back room of the library.",
 "After the storm the beach was covered with driftwood and tangled ropes of kelp.",
 "The narrow bridge over the creek was rebuilt twice in the last forty years.",
 "A thermos of strong coffee is the one thing he never forgets on long drives.",
 "The orchard behind the school produces more apples than the village can eat.",
 "By late autumn the marsh birds have already left for their winter grounds.",
]
def build_filler(tok,n_tokens,offset):
    """neutraler englischer Text mit EXAKT n_tokens Tokens (zyklische Saetze)"""
    if n_tokens<=0: return ""
    txt=""; i=offset
    while len(tok(txt)["input_ids"])<n_tokens+8:
        txt+=SENT_BANK[i%len(SENT_BANK)]+" "; i+=1
    ids=tok(txt)["input_ids"][:n_tokens]
    return tok.decode(ids)
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def think_prefix(u,th=""):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(max(p*(1-p),1e-12)*(1/n1+1/n2))
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
SW=("takeover","gloss","latin-switch(fr)")
def _rj(p):
    rows=[]
    with open(p,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
@torch.no_grad()
def gen_batch(prefix,n,max_new):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       num_return_sequences=n,pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[1]:],skip_special_tokens=True) for o in out]
done=set()
if os.path.exists(OUT_FILL):
    for r in _rj(OUT_FILL): done.add((r["pid"],r["depth"],r["variant"]))
f=open(OUT_FILL,"a",encoding="utf-8")
for pid in PIDS:
    for d in GRID[pid]:
        for v in range(N_VARIANTS if d>0 else N_VARIANTS):
            key=(pid,d,v)
            if key in done: continue
            fill=build_filler(tokenizer,d,offset=v*3)
            cls=[classify_answer(a) for a in gen_batch(think_prefix(PROMPTS[pid],fill),ANS_PER,MAX_NEW)]
            f.write(json.dumps(dict(pid=pid,depth=d,variant=v,cls=cls),ensure_ascii=False)+"\n"); f.flush()
            print("filler %s d=%-5d v%d -> %s"%(pid[:8],d,v,cls))
f.close()

# ---------------- Vergleich echte Kurve vs. Fuellkurve ----------------------
fill_by=collections.defaultdict(list); real_by=collections.defaultdict(list)
_s=set()
for r in _rj(OUT_FILL):
    k=(r["pid"],r["depth"],r["variant"])
    if k in _s: continue
    _s.add(k); fill_by[(r["pid"],r["depth"])].append(sum(1 for c in r["cls"] if c in SW)/len(r["cls"]))
_s=set()
for r in _rj(OUT_ANS):
    k=(r["pid"],r["depth"],r["prefix_idx"])
    if k in _s: continue
    _s.add(k); real_by[(r["pid"],r["depth"])].append(sum(1 for c in r["cls"] if c in SW)/len(r["cls"]))
rng=np.random.default_rng(1)
def ci(fr):
    if not fr: return (float("nan"),)*3
    fr=np.array(fr); b=[float(np.mean(rng.choice(fr,len(fr)))) for _ in range(2000)]
    return float(fr.mean()),float(np.percentile(b,2.5)),float(np.percentile(b,97.5))
fig,axs=plt.subplots(1,len(PIDS),figsize=(6.2*len(PIDS),4.2))
axs=np.atleast_1d(axs)
print("\nVERGLEICH echtes Denken vs. Fuelltext (Anteile ueber Praefixe/Varianten):")
VD={}
for ax,pid in zip(axs,PIDS):
    print("  "+pid[:8]+":")
    ds=sorted(set(d for p,d in fill_by if p==pid))
    for src,by,style in (("echt",real_by,"-o"),("filler",fill_by,"--s")):
        xs,ms,los,his=[],[],[],[]
        for d in ds:
            m,lo,hi=ci(by.get((pid,d),[]))
            if m==m: xs.append(d+1); ms.append(100*m); los.append(100*(m-lo)); his.append(100*(hi-m))
        ax.errorbar(xs,ms,yerr=[los,his],fmt=style,capsize=3,label=src)
    for d in ds:
        fr_r=by_r=real_by.get((pid,d),[]); fr_f=fill_by.get((pid,d),[])
        if fr_r and fr_f:
            kr=int(round(sum(fr_r)*8)); kf=int(round(sum(fr_f)*8))
            print("    d=%-5d echt=%5.1f%%  filler=%5.1f%%  p=%.3f"
                  %(d,100*np.mean(fr_r),100*np.mean(fr_f),twoprop(kr,8*len(fr_r),kf,8*len(fr_f))))
    base=np.mean(real_by.get((pid,0),[0]))
    dstar=256 if pid.startswith("643fdf5d") else 1024
    r_st=np.mean(real_by.get((pid,dstar),[np.nan])); f_st=np.mean(fill_by.get((pid,dstar),[np.nan]))
    if base>0 and r_st==r_st and f_st==f_st and base>r_st:
        share=max(0.0,min(1.5,(base-f_st)/(base-r_st)))
        VD[pid]=share
        print("    -> Verduennungs-Anteil bei d=%d: %.0f%% der echten Unterdrueckung"%(dstar,100*share))
    ax.set_xscale("symlog"); ax.set_title(pid[:8]); ax.set_xlabel("Denk-/Fuell-Tiefe (Tokens, +1)")
    ax.set_ylabel("Switch-Rate (%)"); ax.legend(frameon=False,fontsize=9)
print("\nVERDIKT (Tabellen-Prompt primaer):")
s=VD.get([p for p in PIDS if p.startswith("643fdf5d")][0],None)
if s is None: print("  (unvollstaendige Daten)")
elif s>=0.7: print("  VERDUENNUNG dominiert (%.0f%%): schon themenfremder englischer Text drueckt den"%(100*s))
elif s<=0.3: print("  INHALT dominiert (Verduennung nur %.0f%%): Fuelltext hilft kaum -"%(100*s))
else: print("  MISCHBEFUND (%.0f%% Verduennung): beide Anteile real."%(100*s))
plt.tight_layout(); plt.show()
FILLER_RESULTS=dict(dilution_share=VD)


## Cell 14 — Steering: erst konzentrieren (+α), dann verdünnen (−α)

Extrahiert die Französisch-Moden-Richtung per Teacher-Forcing (8 FR/EN-Paare,
Differenz der Residual-Zustände im mittleren Layerband) und fährt dann den
α-Sweep −2…+2 während der Generierung. +α ist der Kausalbeweis der Richtung,
−α die dosierbare Verdünnung; Degenerations-Guards und köderfreier
Kontroll-Prompt inklusive. Vektoren → Drive (steering_fr.npz).

In [ ]:
# === Cell 14 — STEERING: erst konzentrieren (+a), dann verduennen (-a) ======
# Ersetzt die Kontext-Verduennung (tausende Fuelltokens) durch eine dosierbare
# Vektor-Operation im Aktivierungsraum. Zwei Phasen:
#  1) KONZENTRIEREN = Richtungs-Extraktion: der Franzoesisch-Modus wird per
#     Teacher-Forcing maximal angeworfen (8 FR-Antwortanfaenge vs. 8 englische
#     Zwillinge), Residual-Zustaende am letzten Token -> v_FR = mean(FR)-mean(EN)
#     pro Layer (mittleres Layer-Drittel). Maximales SNR, weil der Modus voll
#     ausgepraegt ist statt aus schwacher natuerlicher Varianz geschaetzt.
#  2) STEERING-SWEEP: waehrend der Generierung wird a*v_FR auf den Residual-
#     Stream addiert, a in [-2..+2] (a=1 = die natuerliche FR/EN-Differenz).
#     +a ist zugleich der KAUSALBEWEIS der Richtung (Rate muss steigen),
#     -a die Verduennung. Guards: Degenerations-Detektor (4-Gramm-Loops),
#     mittlere Antwortlaenge, Beispiele bei |a|=2, plus koederfreier
#     Kontroll-Prompt unter denselben a (Off-Target-Check).
# Grenz-Feinheit: gesteuert werden nur Generierungs-Schritte (rows==batch);
# das erste gesampelte Token entstammt dem ungesteuerten Prefill-Forward.
import json, os, re, math, torch, collections, numpy as np, matplotlib.pyplot as plt
OUT_VEC="/content/drive/MyDrive/weirdspec/steering_fr.npz"
PID=[p for p in PROMPT_IDS if p.startswith("64a66df3")][0]
ALPHAS=[-2.0,-1.0,-0.5,0.0,0.5,1.0,2.0]
N_ANS=32; MAX_NEW=24
FR_BANK=["Voici une grille","Voici la grille structurée","Voici une grille conceptuelle",
 "Bonjour ! Voici la grille","Voici le tableau demandé","Voici une grille écologique",
 "Voici votre grille","Voici une grille 10x10"]
EN_BANK=["Here is a grid","Here is the structured grid","Here is a conceptual grid",
 "Hello! Here is the grid","Here is the requested table","Here is an ecological grid",
 "Here is your grid","Here is a 10x10 grid"]
CONTROL_PROMPT="What should I see in Lisbon in three days? We like food and museums, not so much nightlife."
def think_prefix(u,th=""):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
layers=[m for n,m in model.named_modules() if re.fullmatch(r"model\.layers\.\d+",n)]
if not layers: layers=list(model.model.layers)
NL=len(layers); steer_layers=list(range(NL//3,2*NL//3))
print("layers:",NL,"| steering auf Layer",steer_layers[0],"..",steer_layers[-1])

# ---------------- Phase 1: konzentrieren -> v_FR ----------------------------
CAP={}
def cap_hook(idx):
    def h(mod,inp,out):
        t=out[0] if isinstance(out,tuple) else out
        CAP[idx]=t[0,-1,:].detach().float().cpu()
        return out
    return h
@torch.no_grad()
def last_state(text):
    hs=[l.register_forward_hook(cap_hook(i)) for i,l in enumerate(layers)]
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    model(ids)
    for h in hs: h.remove()
    return {i:CAP[i].clone() for i in range(NL)}
base=think_prefix(PROMPTS[PID],"")
FRs=[last_state(base+s) for s in FR_BANK]
ENs=[last_state(base+s) for s in EN_BANK]
V={}
for i in steer_layers:
    V[i]=torch.stack([s[i] for s in FRs]).mean(0)-torch.stack([s[i] for s in ENs]).mean(0)
np.savez(OUT_VEC,**{str(i):V[i].numpy() for i in steer_layers})
vn=[float(V[i].norm()) for i in steer_layers]
hn=float(torch.stack([s[steer_layers[len(steer_layers)//2]] for s in ENs]).norm(dim=-1).mean())
print("v_FR extrahiert | Norm v: %.1f (Ø Layerband) | Norm h: %.1f -> v/h=%.2f"
      %(sum(vn)/len(vn),hn,(sum(vn)/len(vn))/hn))

# ---------------- Phase 2: Steering-Sweep -----------------------------------
ST={"alpha":0.0,"n_ret":1}
def rows_of(t):
    return int(t.shape[0]*t.shape[1]) if t.dim()==3 else int(t.shape[0])
def steer_hook(idx):
    cache={}
    def h(mod,inp,out):
        t=out[0] if isinstance(out,tuple) else out
        if ST["alpha"]==0.0: return out
        if rows_of(t)!=ST["n_ret"]: return out      # nur Gen-Schritte
        key=(t.device,t.dtype)
        if key not in cache: cache[key]=V[idx].to(device=t.device,dtype=t.dtype)
        t2=t+ST["alpha"]*cache[key]
        return (t2,)+tuple(out[1:]) if isinstance(out,tuple) else t2
    return h
sh=[layers[i].register_forward_hook(steer_hook(i)) for i in steer_layers]
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def rep_loop(t,n=4,k=3):
    w=t.split()
    if len(w)<n: return False
    c=collections.Counter(" ".join(w[i:i+n]) for i in range(len(w)-n+1))
    return max(c.values())>=k
@torch.no_grad()
def gen_batch(prefix,n,max_new):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       num_return_sequences=n,pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[1]:],skip_special_tokens=True) for o in out]
SW=("takeover","gloss","latin-switch(fr)")
RES={}
for name,prompt in (("FR-Grid",PROMPTS[PID]),("Kontrolle",CONTROL_PROMPT)):
    for a in ALPHAS:
        ST.update(alpha=a,n_ret=N_ANS)
        outs=gen_batch(think_prefix(prompt,""),N_ANS,MAX_NEW)
        cls=[classify_answer(x) for x in outs]
        k=sum(1 for c in cls if c in SW); deg=sum(1 for x in outs if rep_loop(x))
        Lm=sum(len(x) for x in outs)/len(outs)
        RES[(name,a)]=(k,N_ANS,deg,Lm)
        p,lo,hi=wilson(k,N_ANS)
        print("%-9s a=%+.1f  switch=%5.1f%% [%4.1f,%4.1f]  degeneriert=%2d/32  OeLen=%.0f"
              %(name,a,100*p,100*lo,100*hi,deg,Lm))
        if abs(a)==2.0: print("    Beispiel:",outs[0][:130].replace("\n"," "))
for h in sh: h.remove()
ST["alpha"]=0.0
fig,ax=plt.subplots(figsize=(7,4))
for name,c in (("FR-Grid","#2563EB"),("Kontrolle","#6B7280")):
    xs=[a for a in ALPHAS]; ys=[100*RES[(name,a)][0]/RES[(name,a)][1] for a in ALPHAS]
    los=[]; his=[]
    for a in ALPHAS:
        k,n,_,_=RES[(name,a)]; p,lo,hi=wilson(k,n); los.append(100*(p-lo)); his.append(100*(hi-p))
    ax.errorbar(xs,ys,yerr=[los,his],marker="o",capsize=3,label=name,color=c)
ax.axvline(0,ls=":",color="black",lw=.8); ax.set_xlabel("Steering-Koeffizient a (x v_FR)")
ax.set_ylabel("Switch-Rate (%)"); ax.legend(frameon=False)
ax.set_title("Steering: konzentrieren (+a) vs. verduennen (-a)")
plt.tight_layout(); plt.show()
kb=RES[("FR-Grid",0.0)][0]; kp=RES[("FR-Grid",2.0)][0]; km=RES[("FR-Grid",-1.0)][0]
degp=RES[("FR-Grid",2.0)][2]; degm=RES[("FR-Grid",-1.0)][2]
print("\nVERDIKT:")
if kp>kb and km<kb and degm<=4:
    print("  RICHTUNG KAUSAL: +a konzentriert (%d->%d), -a verduennt (%d->%d) bei intakter"%(kb,kp,kb,km))
    print("  Fluency (degeneriert: %d/32) - ein Vektor ersetzt die Kontext-Verduennung."%degm)
elif kp<=kb:
    print("  KONZENTRATION FEHLGESCHLAGEN (+a hebt die Rate nicht: %d->%d) -"%(kb,kp))
    print("  v_FR ist nicht die kausale Moden-Richtung (oder falsches Layerband).")
else:
    print("  TEILBEFUND: +a wirkt (%d->%d), aber -a verduennt nicht sauber (%d->%d, deg=%d)."%(kb,kp,kb,km,degm))
STEER_RESULTS={("%s|%+.1f"%k):v for k,v in RES.items()}


## Cell 15 — Hysterese-Test: Rettung aus dem etablierten Modus

Teacher-forced französischer Antwortanfang (k=5/15/40 Tokens, läuft ungesteuert
im Prefill), dann Rettungs-Steering −α·v_FR nur auf neu generierte Tokens
(α=0/0.25/0.5/1.0). Readout: Rettungsrate je (k, α) mit Salat-Guard
(Schriftklassen-Zählung) und FR-Stopwort-Anteil; Heatmap + Verdikt.
Misst die dritte Kostenart: Verlassen nach Lock-in (Kontext-Hysterese).

In [ ]:
# === Cell 15 — HYSTERESE-TEST: Rettung aus dem etablierten Modus ============
# Misst die dritte Kostenart (Verlassen nach Lock-in), die Cell 14 nicht
# messen konnte. Design:
#   * Ein fluessiger franzoesischer Antwortanfang wird TEACHER-FORCED und
#     auf k Tokens geschnitten (k=5/15/40) - er laeuft im PREFILL, also
#     UNGESTEUERT (rows-Regel): die Etablierung des Modus bleibt sauber.
#   * Ab dem ersten neu generierten Token greift -a*v_FR (Rettungs-Dosis),
#     a=0/0.25/0.5/1.0. a=0 ist die Lock-in-Baseline (erwartet: bleibt FR).
#   * Readout je (k,a): Anteil geretteter Fortsetzungen (englisch UND kein
#     Schrift-Salat), dazu FR-Stopwort-Anteil als gradiertes Signal und der
#     Salat-Guard (>=3 Schriftklassen) aus der Cell-14-Lehre.
# Vorhersage (Kontext-Hysterese): Rettungsrate faellt mit k; die noetige
# Dosis waechst mit der Zahl bereits geschriebener FR-Tokens.
import json, os, re, math, torch, collections, numpy as np, matplotlib.pyplot as plt
OUT_VEC="/content/drive/MyDrive/weirdspec/steering_fr.npz"
PID=[p for p in PROMPT_IDS if p.startswith("64a66df3")][0]
K_LIST=[5,15,40]; ALPHAS=[0.0,0.25,0.5,1.0]
N_CONT=16; MAX_NEW=48
FR_ANSWER=("Voici une grille conceptuelle 10×10 illustrant les strates biologiques et les "
 "variables environnementales. Les lignes correspondent aux strates biologiques et les "
 "colonnes aux variables environnementales. Chaque cellule contient une valeur numérique "
 "accompagnée d'une brève description. Le tableau est aligné avec soin pour faciliter la "
 "lecture des données écologiques présentées ci-dessous.")
def think_prefix(u,th=""):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
layers=[m for n,m in model.named_modules() if re.fullmatch(r"model\.layers\.\d+",n)]
if not layers: layers=list(model.model.layers)
if "V" in globals() and isinstance(V,dict) and V:
    VV={int(k):v for k,v in V.items()}
else:
    z=np.load(OUT_VEC); VV={int(k):torch.tensor(z[k]) for k in z.files}
steer_layers=sorted(VV)
print("v_FR geladen | Layerband",steer_layers[0],"..",steer_layers[-1])
ST={"alpha":0.0,"n_ret":1}
def rows_of(t):
    return int(t.shape[0]*t.shape[1]) if t.dim()==3 else int(t.shape[0])
def steer_hook(idx):
    cache={}
    def h(mod,inp,out):
        t=out[0] if isinstance(out,tuple) else out
        if ST["alpha"]==0.0: return out
        if rows_of(t)!=ST["n_ret"]: return out       # nur neu generierte Schritte
        key=(t.device,t.dtype)
        if key not in cache: cache[key]=VV[idx].to(device=t.device,dtype=t.dtype)
        t2=t-ST["alpha"]*cache[key]                   # RETTUNG: minus v_FR
        return (t2,)+tuple(out[1:]) if isinstance(out,tuple) else t2
    return h
sh=[layers[i].register_forward_hook(steer_hook(i)) for i in steer_layers]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont chaque tableau valeur donnees presentees".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by each table value below".split())
def script_classes(t,min_chars=3):
    cls=collections.Counter()
    for ch in t:
        if not ch.isalpha(): continue
        o=ord(ch)
        if o<0x250: cls["latin"]+=1
        elif 0x3040<=o<=0x30FF: cls["kana"]+=1
        elif 0x3400<=o<=0x9FFF or 0xF900<=o<=0xFAFF: cls["cjk"]+=1
        elif 0xAC00<=o<=0xD7AF: cls["hangul"]+=1
        elif 0x0400<=o<=0x052F: cls["cyr"]+=1
        else: cls["other"]+=1
    return sum(1 for v in cls.values() if v>=min_chars)
def classify_cont(t):
    """rescued / french / salat / unklar"""
    if script_classes(t)>=3: return "salat"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    if en>=3 and en>fr: return "rescued"
    if fr>=3 and fr>=en: return "french"
    return "unklar"
def fr_frac(t):
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    if not w: return 0.0
    return sum(1 for x in w if x in FRS)/len(w)
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
@torch.no_grad()
def gen_batch(prefix,n,max_new):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       num_return_sequences=n,pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[1]:],skip_special_tokens=True) for o in out]
fr_ids=tokenizer(FR_ANSWER)["input_ids"]
assert len(fr_ids)>=max(K_LIST), "FR_ANSWER zu kurz fuer max k"
RES={}
for k in K_LIST:
    forced=tokenizer.decode(fr_ids[:k])
    prefix=think_prefix(PROMPTS[PID],"")+forced
    for a in ALPHAS:
        ST.update(alpha=a,n_ret=N_CONT)
        outs=gen_batch(prefix,N_CONT,MAX_NEW)
        cls=[classify_cont(x) for x in outs]
        r=sum(1 for c in cls if c=="rescued"); s=sum(1 for c in cls if c=="salat")
        ff=sum(fr_frac(x) for x in outs)/len(outs)
        RES[(k,a)]=(r,N_CONT,s,ff)
        p,lo,hi=wilson(r,N_CONT)
        print("k=%-3d a=%.2f  gerettet=%5.1f%% [%4.1f,%4.1f]  salat=%d/16  FR-Stopwort-Anteil=%.2f  %s"
              %(k,a,100*p,100*lo,100*hi,s,ff,dict(collections.Counter(cls))))
        if a==1.0: print("    Beispiel:",outs[0][:120].replace("\n"," "))
for h in sh: h.remove()
ST["alpha"]=0.0
M=np.array([[RES[(k,a)][0]/RES[(k,a)][1] for a in ALPHAS] for k in K_LIST])
fig,ax=plt.subplots(figsize=(6.4,4))
im=ax.imshow(100*M,cmap="RdYlGn",vmin=0,vmax=100,aspect="auto")
ax.set_xticks(range(len(ALPHAS))); ax.set_xticklabels(["%.2f"%a for a in ALPHAS])
ax.set_yticks(range(len(K_LIST))); ax.set_yticklabels(["k=%d"%k for k in K_LIST])
for i in range(len(K_LIST)):
    for j in range(len(ALPHAS)):
        ax.text(j,i,"%.0f%%"%(100*M[i,j]),ha="center",va="center",fontsize=10)
ax.set_xlabel("Rettungs-Dosis a (x v_FR, subtraktiv)"); ax.set_ylabel("etablierte FR-Tokens")
ax.set_title("Hysterese: Rettungsrate aus dem etablierten Modus")
plt.colorbar(im,ax=ax,fraction=0.045); plt.tight_layout(); plt.show()
print("\nVERDIKT:")
base_lock=[RES[(k,0.0)][0] for k in K_LIST]
if max(base_lock)<=N_CONT*0.25:
    print("  Lock-in-Baseline bestaetigt (a=0 bleibt ueberwiegend franzoesisch).")
else:
    print("  !! Lock-in-Baseline schwach (a=0 rettet %s/16) - Etablierung pruefen."%base_lock)
drops=sum(1 for a in ALPHAS[1:] if RES[(K_LIST[0],a)][0]>RES[(K_LIST[-1],a)][0])
if drops>=2:
    print("  HYSTERESE: Rettungsrate faellt mit k bei fester Dosis - der etablierte")
    print("  Kontext traegt den Modus; die noetige Dosis waechst mit k.")
elif all(RES[(k,a)][0]>=N_CONT*0.75 for k in K_LIST for a in ALPHAS[1:]):
    print("  KEINE HYSTERESE: schon kleine Dosen retten unabhaengig von k -")
    print("  der Zustand dominiert den Kontext; Lock-in ist steering-fragil.")
else:
    print("  MISCHBILD - Matrix oben ansehen (Salat-Spalte beachten).")
RESCUE_RESULTS={("k%d|a%.2f"%k):v for k,v in RES.items()}


## Cell 16 — Form der Rettungskurve + Satzgrenzen-Test

Feines k-Gitter (5–60) bei α=0.25 zeigt, ob der Hysterese-Abfall glatt oder
treppenförmig ist; der Mikroscan schneidet denselben Text bei t₁−2/t₁/t₁+2
Tokens um das erste Satzende — springt die Rettungsrate über 4 Tokens genau
an der Grenze, akkumuliert Commitment pro abgeschlossenem Satz
(ganzzahliges Quantum). Satzgrenzen offset-basiert, prefix-treu bestimmt.

In [ ]:
# === Cell 16 — FORM DER RETTUNGSKURVE + SATZGRENZEN-TEST ====================
# Zwei Fragen an die Hysterese aus Cell 15:
#  A) FEINKURVE: Wie faellt die Rettungsrate mit k ab - glatt (logistisch in
#     der Token-Zahl) oder als TREPPE an Diskursgrenzen? k-Gitter 5..60 bei
#     der sauberen Dosis a=0.25.
#  B) SATZGRENZEN-MIKROSCAN: derselbe Text, geschnitten bei t1-2 / t1 / t1+2
#     Tokens (t1 = Ende des ersten Satzes). Springt die Rate ueber 4 Tokens
#     hinweg genau an der Grenze, akkumuliert Commitment pro ABGESCHLOSSENEM
#     Satz (ganzzahliges Quantum, floor-Funktion auf der Diskurs-Achse);
#     bleibt sie stetig, zaehlt die Token-Masse.
# Reuse: v_FR (Session oder Drive), Steering subtraktiv nur auf neue Tokens.
import json, os, re, math, torch, collections, numpy as np, matplotlib.pyplot as plt
OUT_VEC="/content/drive/MyDrive/weirdspec/steering_fr.npz"
PID=[p for p in PROMPT_IDS if p.startswith("64a66df3")][0]
ALPHA=0.25; K_GRID=[5,10,15,20,25,30,35,40,60]; N_A=16; N_B=24; MAX_NEW=48
FR_ANSWER=("Voici une grille conceptuelle 10×10 illustrant les strates biologiques et les "
 "variables environnementales. Les lignes correspondent aux strates biologiques et les "
 "colonnes aux variables environnementales. Chaque cellule contient une valeur numérique "
 "accompagnée d'une brève description. Le tableau est aligné avec soin pour faciliter la "
 "lecture des données écologiques présentées ci-dessous. Les valeurs sont exprimées dans "
 "des unités cohérentes et chaque colonne est clairement étiquetée.")
def think_prefix(u,th=""):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
def sent_boundaries(text,tok):
    """Token-Indizes NACH jedem Satzende (Offset-basiert, prefix-treu)"""
    enc=tok(text,return_offsets_mapping=True)
    ids=enc["input_ids"]; offs=enc["offset_mapping"]
    bounds=[]
    for m in re.finditer(r"\.",text):
        cpos=m.start()
        for i,(s,e) in enumerate(offs):
            if e>cpos: bounds.append(i+1); break
    return ids,sorted(set(bounds))
layers=[m for n,m in model.named_modules() if re.fullmatch(r"model\.layers\.\d+",n)]
if not layers: layers=list(model.model.layers)
if "VV" in globals() and isinstance(VV,dict) and VV:
    pass
elif "V" in globals() and isinstance(V,dict) and V:
    VV={int(k):v for k,v in V.items()}
else:
    z=np.load(OUT_VEC); VV={int(k):torch.tensor(z[k]) for k in z.files}
steer_layers=sorted(VV)
ST={"alpha":0.0,"n_ret":1}
def rows_of(t): return int(t.shape[0]*t.shape[1]) if t.dim()==3 else int(t.shape[0])
def steer_hook(idx):
    cache={}
    def h(mod,inp,out):
        t=out[0] if isinstance(out,tuple) else out
        if ST["alpha"]==0.0: return out
        if rows_of(t)!=ST["n_ret"]: return out
        key=(t.device,t.dtype)
        if key not in cache: cache[key]=VV[idx].to(device=t.device,dtype=t.dtype)
        t2=t-ST["alpha"]*cache[key]
        return (t2,)+tuple(out[1:]) if isinstance(out,tuple) else t2
    return h
sh=[layers[i].register_forward_hook(steer_hook(i)) for i in steer_layers]
FRSW=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont chaque tableau valeur donnees presentees unites colonne".split())
ENSW=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by each table value below".split())
def script_classes(t,min_chars=3):
    cls=collections.Counter()
    for ch in t:
        if not ch.isalpha(): continue
        o=ord(ch)
        if o<0x250: cls["latin"]+=1
        elif 0x3040<=o<=0x30FF: cls["kana"]+=1
        elif 0x3400<=o<=0x9FFF or 0xF900<=o<=0xFAFF: cls["cjk"]+=1
        elif 0xAC00<=o<=0xD7AF: cls["hangul"]+=1
        elif 0x0400<=o<=0x052F: cls["cyr"]+=1
        else: cls["other"]+=1
    return sum(1 for v in cls.values() if v>=min_chars)
def classify_cont(t):
    if script_classes(t)>=3: return "salat"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRSW); en=sum(1 for x in w if x in ENSW)
    if en>=3 and en>fr: return "rescued"
    if fr>=3 and fr>=en: return "french"
    return "unklar"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(max(p*(1-p),1e-12)*(1/n1+1/n2))
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
@torch.no_grad()
def gen_batch(prefix,n,max_new):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       num_return_sequences=n,pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[1]:],skip_special_tokens=True) for o in out]
ids,bounds=sent_boundaries(FR_ANSWER,tokenizer)
t1=bounds[0]
print("Passage: %d Tokens | Satzgrenzen nach Token %s | t1=%d"%(len(ids),bounds,t1))
def rescue_at(k,n):
    forced=tokenizer.decode(ids[:k])
    ST.update(alpha=ALPHA,n_ret=n)
    outs=gen_batch(think_prefix(PROMPTS[PID],"")+forced,n,MAX_NEW)
    cls=[classify_cont(x) for x in outs]
    return sum(1 for c in cls if c=="rescued"),n,dict(collections.Counter(cls))
print("\nA) FEINKURVE (a=%.2f):"%ALPHA)
CURVE={}
for k in K_GRID:
    if k>len(ids): continue
    r,n,cc=rescue_at(k,N_A); CURVE[k]=(r,n,cc)
    p,lo,hi=wilson(r,n)
    mark=" <- SATZGRENZE" if any(abs(k-b)<=1 for b in bounds) else ""
    print("  k=%-3d gerettet=%5.1f%% [%4.1f,%4.1f]  %s%s"%(k,100*p,100*lo,100*hi,cc,mark))
print("\nB) MIKROSCAN um die erste Satzgrenze (t1=%d):"%t1)
MICRO={}
for k in (max(1,t1-2),t1,t1+2):
    r,n,cc=rescue_at(k,N_B); MICRO[k]=(r,n,cc)
    p,lo,hi=wilson(r,n)
    print("  k=%-3d (%s Grenze)  gerettet=%5.1f%% [%4.1f,%4.1f]  %s"
          %(k,"vor" if k<t1 else("an" if k==t1 else "nach"),100*p,100*lo,100*hi,cc))
ST["alpha"]=0.0
for h in sh: h.remove()
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
ks=sorted(CURVE); ys=[100*CURVE[k][0]/CURVE[k][1] for k in ks]
los=[]; his=[]
for k in ks:
    r,n,_=CURVE[k]; p,lo,hi=wilson(r,n); los.append(100*(p-lo)); his.append(100*(hi-p))
ax[0].errorbar(ks,ys,yerr=[los,his],marker="o",capsize=3,color="#2563EB")
for b in bounds:
    if b<=max(ks): ax[0].axvline(b,ls=":",color="#DC2626",lw=1)
ax[0].set_xlabel("etablierte FR-Tokens k"); ax[0].set_ylabel("Rettungsrate (%)")
ax[0].set_title("Feinkurve (rot: Satzgrenzen)")
mk=sorted(MICRO); my=[100*MICRO[k][0]/MICRO[k][1] for k in mk]
mlo=[]; mhi=[]
for k in mk:
    r,n,_=MICRO[k]; p,lo,hi=wilson(r,n); mlo.append(100*(p-lo)); mhi.append(100*(hi-p))
ax[1].bar(range(len(mk)),my,yerr=[mlo,mhi],capsize=4,color=["#10B981","#F59E0B","#DC2626"])
ax[1].set_xticks(range(len(mk))); ax[1].set_xticklabels(["k=%d\n(vor)"%mk[0],"k=%d\n(an)"%mk[1],"k=%d\n(nach)"%mk[2]])
ax[1].set_ylabel("Rettungsrate (%)"); ax[1].set_title("Mikroscan: 4 Tokens um die Satzgrenze")
plt.tight_layout(); plt.show()
k_pre,k_post=max(1,t1-2),t1+2
r1,n1,_=MICRO[k_pre]; r2,n2,_=MICRO[k_post]
pv=twoprop(r1,n1,r2,n2); drop=r1/n1-r2/n2
print("\nVERDIKT Mikroscan: vor %.0f%% vs. nach %.0f%% | Sprung=%.0fpp | p=%.3f"
      %(100*r1/n1,100*r2/n2,100*drop,pv))
if pv<0.05 and drop>=0.25:
    print("  TREPPE: Der Ausstiegspreis springt AN der Satzgrenze - Commitment")
    print("  akkumuliert pro abgeschlossenem Satz (ganzzahliges Quantum).")
elif pv>=0.3 and abs(drop)<0.15:
    print("  GLATT an der Grenze: kein Satz-Quantum - die Token-Masse zaehlt;")
    print("  Kurvenform links in der Feinkurve ablesen.")
else:
    print("  UNKLAR/schwach - Feinkurve und CIs ansehen; ggf. N erhoehen.")
CURVE_RESULTS=dict(curve={int(k):v[:2] for k,v in CURVE.items()},
                   micro={int(k):v[:2] for k,v in MICRO.items()},t1=int(t1),bounds=[int(b) for b in bounds])


## Cell 17 — Druckprofil: das Austrittsbedürfnis, positionsaufgelöst

Teacher-forced englische Antwort; an jeder Position wird die unterdrückte
Fremd-Wahrscheinlichkeitsmasse abgelesen (Vokabular-Masken: Fremdschrift +
Akzent-Latein als FR-Proxy). Tore (Antwortstart, Satzenden, Zell-/Zeilen-
grenzen) markiert; Permutationstest auf Tor-Effekt, Hüllkurven-Klassifikation
(abklingend = prompt-induziert vs. flach = strukturell). Drei Prompts:
JP-Tabelle, FR-Grid, Sofort-Kipper.

In [ ]:
# === Cell 17 — DRUCKPROFIL: das Austrittsbeduerfnis, positionsaufgeloest ====
# Trennt DRUCK von GELEGENHEIT: Eine ENGLISCHE Antwort wird teacher-forced,
# und an jeder Position wird die Wahrscheinlichkeitsmasse des unterdrueckten
# fremden Zweigs abgelesen (naechstes-Token-Verteilung). Das ergibt das
# Druckprofil ueber die Antwort, mit den TOREN markiert (Antwortstart,
# Satzenden, Zell-/Zeilengrenzen). Fragen:
#   * TOR-EFFEKT: spitzt der Druck an den Toren? (Permutationstest)
#   * HUELLKURVE: klingt der Druck mit dem Abstand vom Prompt ab
#     (prompt-induziert) oder bleibt er flach (strukturell)?
# Zwei Masken: M_script (nicht-lateinische Tokens - CJK/Kana/...) und
# M_akzent (akzentuiertes Latein als grober FR-Proxy; ehrlicher Vorbehalt:
# viele franzoesische Tokens sind akzentfrei, der FR-Druck wird unterschaetzt).
# Prompts: JP-Tabelle, FR-Grid, Sofort-Kipper (64e10d84, aus weird_transcripts).
import json, os, re, glob, math, torch, collections, numpy as np, matplotlib.pyplot as plt
DATA_DIR=globals().get("DATA_DIR","/content/drive/MyDrive/weirdspec")
MASK_PATH="/content/drive/MyDrive/weirdspec/vocab_foreign_masks.npz"
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def _rj(p):
    rows=[]
    with open(p,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def _ff(d,*names):
    for nm in names:
        q=os.path.join(d,nm)
        if os.path.isfile(q): return q
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None
# ---- Prompts einsammeln (inkl. Sofort-Kipper aus weird_transcripts) --------
P={}
P["JP-Tabelle"]=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
P["FR-Grid"]  =PROMPTS[[p for p in PROMPT_IDS if p.startswith("64a66df3")][0]]
wp=_ff(DATA_DIR,"weird_transcripts.jsonl")
for l in open(wp,encoding="utf-8"):
    r=json.loads(l)
    if r["id"].startswith("64e10d84ba"):
        P["Sofort"]=next(t["content"] for t in r["conversations"] if t["role"]=="user"); break
EN={
 "JP-Tabelle":("| Service Name | Storage Limit |\n|---|---|\n"
   "| Google Drive | Google Drive offers 15 GB of free storage for every account. |\n"
   "| Dropbox | Dropbox provides 2 GB of free storage on its basic plan. |\n"
   "| OneDrive | OneDrive includes 5 GB of free storage with a Microsoft account. |"),
 "FR-Grid":("Here is a conceptual 10x10 grid. The rows correspond to biological strata "
   "and the columns to environmental variables. Each cell contains a value and a brief "
   "description. The table is aligned carefully so the data can be read easily."),
 "Sofort":("Of course, I would be happy to help you with that. Please tell me exactly "
   "what you need and I will prepare it right away. We can then refine the result "
   "together step by step until it matches your expectations."),
}
def think_prefix(u,th=""):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---- Vokabular-Masken (einmalig, gecacht) ----------------------------------
def build_masks(tok,path):
    if os.path.exists(path):
        z=np.load(path); return torch.tensor(z["script"]),torch.tensor(z["acc"])
    V=len(tok); ms=np.zeros(V,bool); ma=np.zeros(V,bool); B=4096
    for s in range(0,V,B):
        texts=tok.batch_decode([[i] for i in range(s,min(V,s+B))])
        for j,tx in enumerate(texts):
            for ch in tx:
                o=ord(ch)
                if ch.isalpha() and o>=0x250 and any(a<=o<=b for a,b in FRW): ms[s+j]=True; break
                if 0x00C0<=o<=0x017F: ma[s+j]=True
    np.savez(path,script=ms,acc=ma); return torch.tensor(ms),torch.tensor(ma)
M_script,M_acc=build_masks(tokenizer,MASK_PATH)
print("Masken: script=%d Tokens | akzent=%d Tokens"%(int(M_script.sum()),int(M_acc.sum())))
# ---- Tore aus dem Antworttext (offset-basiert, prefix-treu) ----------------
def gates_of(full,a_char,tok):
    enc=tok(full,return_offsets_mapping=True)
    ids=enc["input_ids"]; offs=enc["offset_mapping"]
    a_tok=next(i for i,(s,e) in enumerate(offs) if s>=a_char and e>s)
    gates={0}
    for m in re.finditer(r"[.|\n]",full[a_char:]):
        cpos=a_char+m.start()
        for i,(s,e) in enumerate(offs):
            if e>cpos:
                gi=i+1-a_tok
                if gi>0: gates.add(gi)
                break
    return ids,a_tok,sorted(gates)
# ---- Druckprofil -----------------------------------------------------------
@torch.no_grad()
def profile(user,answer):
    pre=think_prefix(user,""); full=pre+answer
    ids,a_tok,gates=gates_of(full,len(pre),tokenizer)
    t=torch.tensor([ids],device=model.device)
    logits=model(t).logits[0]
    V=logits.shape[-1]                      # Modell-Vokabular kann gepolstert sein
    def _pad(m):
        m=m.to(model.device)
        if m.shape[0]<V:
            m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
        return m[:V]
    Ms=_pad(M_script); Ma=_pad(M_acc)
    fs=[]; fa=[]
    for pos in range(a_tok-1,len(ids)-1):
        p=torch.softmax(logits[pos].float(),-1)
        fs.append(float(p[Ms].sum())); fa.append(float(p[Ma].sum()))
    gates=[g for g in gates if g<len(fs)]
    return np.array(fs),np.array(fa),gates
def perm_p(f,gates,nperm=2000,seed=0):
    g=np.zeros(len(f),bool); g[gates]=True
    if g.sum()==0 or (~g).sum()==0: return float("nan"),0.0
    obs=f[g].mean()-f[~g].mean()
    rng=np.random.default_rng(seed); cnt=0
    for _ in range(nperm):
        rp=rng.permutation(len(f))
        gp=np.zeros(len(f),bool); gp[rp[:g.sum()]]=True
        if f[gp].mean()-f[~gp].mean()>=obs: cnt+=1
    return (1+cnt)/(1+nperm),obs
fig,axs=plt.subplots(1,len(P),figsize=(6*len(P),4))
axs=np.atleast_1d(axs)
print()
for ax,(name,user) in zip(axs,P.items()):
    fs,fa,gates=profile(user,EN[name])
    x=np.arange(len(fs))
    ax.plot(x,100*fs,color="#2563EB",label="Fremdschrift-Masse")
    ax.plot(x,100*fa,color="#10B981",alpha=0.7,label="Akzent-Latein (FR-Proxy)")
    for g in gates: ax.axvline(g,ls=":",color="#DC2626",lw=.8)
    ax.set_yscale("log"); ax.set_xlabel("Antwort-Token-Position"); ax.set_ylabel("Druck (%)")
    ax.set_title(name); ax.legend(frameon=False,fontsize=8)
    p_s,d_s=perm_p(fs,gates); p_a,d_a=perm_p(fa,gates,seed=1)
    third=max(1,len(fs)//3)
    env_s=fs[:third].mean()/max(fs[-third:].mean(),1e-9)
    print("%-10s Tore=%d | TOR-EFFEKT script: Delta=%+.4f p=%.3f | akzent: Delta=%+.4f p=%.3f"
          %(name,len(gates),d_s,p_s,d_a,p_a))
    print("           HUELLKURVE (erstes/letztes Drittel, script): %.2fx -> %s"
          %(env_s,"ABKLINGEND (prompt-induziert)" if env_s>2 else
             ("ANSTEIGEND" if env_s<0.5 else "FLACH (eher strukturell/konstant gehalten)")))
    print("           Spitzen-Druck: script max=%.2f%% @pos %d | Start-Tor=%.2f%%"
          %(100*fs.max(),int(fs.argmax()),100*fs[0]))
plt.tight_layout(); plt.show()
print("\nLESART: Tor-Effekt p<0.05 => der Druck sammelt sich an den Sinneinheiten-")
print("Grenzen (Gelegenheits-Theorie bestaetigt, jetzt logit-basiert statt ueber")
print("Kipp-Raten). Huellkurve abklingend => Druck ist prompt-induziert und verbraucht")
print("sich; flach => konstante Quelle (Struktur oder dauerhaft praesenter Koeder).")


## Cell 18 — Geometrie der drei Kategorien

1) Sprach-Kegel: v_JP/KR/ZH/FR im selben Kontext extrahiert → Kosinus-Matrix,
gemeinsame Komponente, effektive Dimension (PR). 2) Feder vs. Fahrtrichtung:
Dispositions-Richtung (Köder vs. bereinigte Paraphrase, Prompt-End-Zustände)
gegen die Modus-Richtungen. 3) Tor-Breite: Entropie-Profil entlang forcierter
Antworten — weitet sich der erreichbare Raum an Sinneinheiten-Grenzen?

In [ ]:
# === Cell 18 — GEOMETRIE der drei Kategorien ================================
# Drei falsifizierbare Messungen statt Einzel-Achsen-Glaube:
#  1) SPRACH-KEGEL: Richtungen v_JP/v_KR/v_ZH/v_FR per Forced-Prefix im
#     SELBEN Kontext (JP-Tabellen-Prompt, Tabellenkopf in 5 Sprachen, nur die
#     Sprache variiert) -> Kosinus-Matrix, gemeinsame Komponente, effektive
#     Dimension (Partizipations-Ratio der Gram-Eigenwerte).
#     Vorhersage: Schrift-Richtungen kohaerent, v_FR nahezu senkrecht dazu.
#  2) FEDER vs. FAHRTRICHTUNG: Dispositions-Richtung aus Prompt-End-Zustaenden
#     (Koeder-Prompt vs. minimal bereinigte Paraphrase: "local name"->"name")
#     vs. Modus-Richtungen. Kosinus ~0 -> Druck und Modus sind getrennte
#     geometrische Objekte.
#  3) TOR-BREITE: Naechstes-Token-Entropie entlang teacher-forced Antworten
#     (EN-Tabelle + FR-Passage) - weitet sich der erreichbare Raum an den
#     Sinneinheiten-Grenzen? (Permutationstest; erklaert den Saegezahn
#     geometrisch: Rettung ist leicht, wo der Kanal breit ist.)
import json, os, re, math, torch, collections, numpy as np, matplotlib.pyplot as plt
def think_prefix(u,th=""):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
layers=[m for n,m in model.named_modules() if re.fullmatch(r"model\.layers\.\d+",n)]
if not layers: layers=list(model.model.layers)
NL=len(layers); BAND=list(range(NL//3,2*NL//3))
CAP={}
def cap_hook(idx):
    def h(mod,inp,out):
        t=out[0] if isinstance(out,tuple) else out
        CAP[idx]=t[0,-1,:].detach().float().cpu()
        return out
    return h
@torch.no_grad()
def last_state(text):
    hs=[layers[i].register_forward_hook(cap_hook(i)) for i in BAND]
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    model(ids)
    for h in hs: h.remove()
    return {i:CAP[i].clone() for i in BAND}
def dirvec(A,B):
    """mean(A)-mean(B), pro Layer normiert, konkateniert -> eine Richtung"""
    vs=[]
    for l in BAND:
        a=torch.stack([s[l] for s in A]).mean(0); b=torch.stack([s[l] for s in B]).mean(0)
        d=a-b; vs.append(d/(d.norm()+1e-8))
    v=torch.cat(vs); return v/v.norm()
def cosm(vecs):
    names=list(vecs); M=np.zeros((len(names),len(names)))
    for i,a in enumerate(names):
        for j,b in enumerate(names):
            M[i,j]=float(torch.dot(vecs[a],vecs[b]))
    return names,M
def pr_dim(G):
    lam=np.linalg.eigvalsh(G); lam=np.clip(lam,0,None)
    return float(lam.sum()**2/max((lam**2).sum(),1e-12))

# ---------------- 1) Sprach-Kegel -------------------------------------------
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
base=think_prefix(TAB,"")
COL2={"EN":["Storage Limit","Storage Capacity","Storage Allowance","Free Storage"],
      "JP":["ストレージ制限","保存容量","容量制限","ストレージ上限"],
      "KR":["저장 용량","저장 한도","용량 제한","저장 공간"],
      "ZH":["存储限制","存储容量","容量上限","存储空间"],
      "FR":["Limite de stockage","Capacité de stockage","Espace de stockage","Quota de stockage"]}
COL1={"EN":"Service Name","JP":"サービス名","KR":"서비스명","ZH":"服务名称","FR":"Nom du service"}
def bank(lang):
    out=[]
    for c2 in COL2[lang]:
        h="| %s | %s |"%(COL1[lang],c2)
        out+=[h,h+"\n|---|---|"]
    return out                      # 8 Varianten
STATES={L:[last_state(base+s) for s in bank(L)] for L in COL2}
DIRS={L:dirvec(STATES[L],STATES["EN"]) for L in COL2 if L!="EN"}
names,M=cosm(DIRS)
print("1) SPRACH-KEGEL — Kosinus-Matrix (gleicher Kontext, gleiche Struktur):")
print("      "+"  ".join("%5s"%n for n in names))
for i,n in enumerate(names):
    print("  %3s "%n+"  ".join("%5.2f"%M[i,j] for j in range(len(names))))
common=torch.stack(list(DIRS.values())).mean(0); common=common/common.norm()
print("  cos zur gemeinsamen Komponente: "+
      "  ".join("%s=%.2f"%(n,float(torch.dot(DIRS[n],common))) for n in names))
print("  effektive Dimension (PR der Gram-Matrix): %.2f von %d"%(pr_dim(M),len(names)))

# ---------------- 2) Feder (Druck) vs. Fahrtrichtung (Modus) ----------------
if "local name" in TAB:
    CLEAN=TAB.replace("local name","name")
else:
    CLEAN=TAB.replace("local","")
    print("  (Warnung: 'local name' nicht woertlich im Prompt - generischer Ersatz)")
SUF=[""," Thanks."," Please."," Thank you!"," Thanks in advance."," Best regards.",
     " Appreciated."," Cheers."]
S_dec=[last_state(think_prefix(TAB+s,"")) for s in SUF]
S_cln=[last_state(think_prefix(CLEAN+s,"")) for s in SUF]
v_druck=dirvec(S_dec,S_cln)
print("\n2) FEDER vs. FAHRTRICHTUNG:")
for n in names:
    print("  cos(v_druck, v_%s) = %+.2f"%(n,float(torch.dot(v_druck,DIRS[n]))))
print("  cos(v_druck, common) = %+.2f"%float(torch.dot(v_druck,common)))
print("  (|cos|<~0.2: Druck und Modus sind getrennte geometrische Objekte;")
print("   hoch: eine Achse an zwei Orten.)")

# ---------------- 3) Tor-Breite (Entropie-Profil) ---------------------------
def gates_of(full,a_char,tok):
    enc=tok(full,return_offsets_mapping=True)
    ids=enc["input_ids"]; offs=enc["offset_mapping"]
    a_tok=next(i for i,(s,e) in enumerate(offs) if s>=a_char and e>s)
    gates={0}
    for m in re.finditer(r"[.|\n]",full[a_char:]):
        cpos=a_char+m.start()
        for i,(s,e) in enumerate(offs):
            if e>cpos:
                gi=i+1-a_tok
                if gi>0: gates.add(gi)
                break
    return ids,a_tok,sorted(gates)
@torch.no_grad()
def entropy_profile(user,answer):
    pre=think_prefix(user,""); full=pre+answer
    ids,a_tok,gates=gates_of(full,len(pre),tokenizer)
    logits=model(torch.tensor([ids],device=model.device)).logits[0]
    H=[]
    for pos in range(a_tok-1,len(ids)-1):
        p=torch.softmax(logits[pos].float(),-1)
        H.append(float(-(p*torch.log(p+1e-12)).sum()))
    return np.array(H),[g for g in gates if g<len(H)]
def perm_p(f,gates,nperm=2000,seed=0):
    g=np.zeros(len(f),bool); g[gates]=True
    if g.sum()==0 or (~g).sum()==0: return float("nan"),0.0
    obs=f[g].mean()-f[~g].mean()
    rng=np.random.default_rng(seed); cnt=0
    for _ in range(nperm):
        rp=rng.permutation(len(f))
        gp=np.zeros(len(f),bool); gp[rp[:g.sum()]]=True
        if f[gp].mean()-f[~gp].mean()>=obs: cnt+=1
    return (1+cnt)/(1+nperm),obs
EN_TAB=("| Service Name | Storage Limit |\n|---|---|\n"
 "| Google Drive | Google Drive offers 15 GB of free storage for every account. |\n"
 "| Dropbox | Dropbox provides 2 GB of free storage on its basic plan. |\n"
 "| OneDrive | OneDrive includes 5 GB of free storage with a Microsoft account. |")
FRP=PROMPTS[[p for p in PROMPT_IDS if p.startswith("64a66df3")][0]]
FR_ANS=("Voici une grille conceptuelle 10×10 illustrant les strates biologiques et les "
 "variables environnementales. Les lignes correspondent aux strates biologiques et les "
 "colonnes aux variables environnementales. Chaque cellule contient une valeur numérique "
 "accompagnée d'une brève description.")
print("\n3) TOR-BREITE (naechstes-Token-Entropie an Toren vs. in Einheiten):")
fig,axs=plt.subplots(1,2,figsize=(12,4))
for ax,(nm,u,a) in zip(axs,[("EN-Tabelle",TAB,EN_TAB),("FR-Passage",FRP,FR_ANS)]):
    H,g=entropy_profile(u,a)
    pv,d=perm_p(H,g)
    ax.plot(H,color="#2563EB")
    for gi in g: ax.axvline(gi,ls=":",color="#DC2626",lw=.8)
    ax.set_title("%s  (Delta=%.2f nats, p=%.3f)"%(nm,d,pv))
    ax.set_xlabel("Antwort-Token-Position"); ax.set_ylabel("Entropie (nats)")
    print("  %-11s Tore=%d | Entropie an Toren %.2f vs. in Einheiten %.2f | Delta=%+.2f p=%.3f"
          %(nm,len(g),H[np.isin(np.arange(len(H)),g)].mean(),
            H[~np.isin(np.arange(len(H)),g)].mean(),d,pv))
plt.tight_layout(); plt.show()
print("\nLESART: 1) PR>>1 + FR-Orthogonalitaet = Kegel statt Achse. 2) cos~0 =")
print("Druck (Feder im Prompt-Zustand) und Modus (Fahrtrichtung der Generierung)")
print("sind verschiedene Objekte. 3) Delta>0, p<0.05 = Tore sind Aufweitungen des")
print("erreichbaren Raums - die geometrische Erklaerung des Saegezahns.")


## Cell 19 — Feder-Injektion + FR-Tor-Power

A) Kausaltest der Druck-Richtung: die Feder wird dem bereinigten Prompt an
exakt ihrem Extraktions-Ort injiziert (letzte Prompt-Position, nur Prefill),
Dosis 0/0.5/1/2; Referenz Köder-Prompt, Neutral-Kontrolle mit voller Feder.
B) Entropie-Tor-Test mit 9-Satz-FR-Passage (ausreichend Tore für Power).

In [ ]:
# === Cell 19 — FEDER-INJEKTION + FR-TOR-POWER ==============================
# A) FEDER-INJEKTION (Kausaltest der Druck-Richtung): v_druck wurde aus dem
#    Minimalpaar Koeder/bereinigt extrahiert (Cell 18). Jetzt wird die Feder
#    dem BEREINIGTEN Prompt injiziert - additiv an EXAKT ihrem Extraktions-
#    Ort: letzte Prompt-Position, nur waehrend des Prefills (Gen-Schritte
#    unberuehrt). Dosis a in {0, 0.5, 1, 2} (a=1 = die natuerliche Differenz).
#    Steigt die Kipp-Rate Richtung Koeder-Baseline, ist der Druck kausal:
#    der Koeder waere durch einen Vektor ersetzt (synthetische Disposition).
#    Kontrollen: Koeder-Prompt (Referenz), neutraler Prompt + volle Feder
#    (ist die Feder aufgaben-kontextuell oder ein globaler Kipp-Knopf?).
# B) FR-TOR-POWER: lange franzoesische Passage (9 Saetze -> ~9 Tore) fuer
#    den Entropie-Tor-Test mit ausreichender Statistik.
import json, os, re, math, torch, collections, numpy as np, matplotlib.pyplot as plt
def think_prefix(u,th=""):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
layers=[m for n,m in model.named_modules() if re.fullmatch(r"model\.layers\.\d+",n)]
if not layers: layers=list(model.model.layers)
NL=len(layers); BAND=list(range(NL//3,2*NL//3))
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
CLEAN=TAB.replace("local name","name") if "local name" in TAB else TAB.replace("local","")
NEUTRAL="What should I see in Lisbon in three days? We like food and museums, not so much nightlife."
N_ANS=32; MAX_NEW=24; ALPHAS=[0.0,0.5,1.0,2.0]
# ---- Feder (roh, pro Layer) aus Cell-18-Zustaenden oder frisch extrahieren -
CAP={}
def cap_hook(idx):
    def h(mod,inp,out):
        t=out[0] if isinstance(out,tuple) else out
        CAP[idx]=t[0,-1,:].detach().float().cpu()
        return out
    return h
@torch.no_grad()
def last_state(text):
    hs=[layers[i].register_forward_hook(cap_hook(i)) for i in BAND]
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    model(ids)
    for h in hs: h.remove()
    return {i:CAP[i].clone() for i in BAND}
if "S_dec" in globals() and "S_cln" in globals():
    _Sd,_Sc=S_dec,S_cln
else:
    SUF=[""," Thanks."," Please."," Thank you!"," Thanks in advance."," Best regards.",
         " Appreciated."," Cheers."]
    _Sd=[last_state(think_prefix(TAB+s,"")) for s in SUF]
    _Sc=[last_state(think_prefix(CLEAN+s,"")) for s in SUF]
SPRING={l:(torch.stack([s[l] for s in _Sd]).mean(0)-torch.stack([s[l] for s in _Sc]).mean(0))
        for l in BAND}
sn=[float(SPRING[l].norm()) for l in BAND]
print("Feder extrahiert | Norm pro Layer: median %.2f (min %.2f, max %.2f)"
      %(float(np.median(sn)),min(sn),max(sn)))
SP={"alpha":0.0,"n_ret":1}
def spring_hook(idx):
    cache={}
    def h(mod,inp,out):
        t=out[0] if isinstance(out,tuple) else out
        if SP["alpha"]==0.0: return out
        rows=int(t.shape[0]*t.shape[1]) if t.dim()==3 else int(t.shape[0])
        if rows==SP["n_ret"]: return out           # Gen-Schritte unberuehrt
        key=(t.device,t.dtype)
        if key not in cache: cache[key]=SPRING[idx].to(device=t.device,dtype=t.dtype)
        t2=t.clone(); t2[:,-1,:]=t2[:,-1,:]+SP["alpha"]*cache[key]
        return (t2,)+tuple(out[1:]) if isinstance(out,tuple) else t2
    return h
sh=[layers[i].register_forward_hook(spring_hook(i)) for i in BAND]
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
@torch.no_grad()
def gen_batch(prefix,n,max_new):
    ids=tokenizer(prefix,return_tensors="pt").input_ids.to(model.device)
    out=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                       repetition_penalty=1.0,max_new_tokens=max_new,
                       num_return_sequences=n,pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[1]:],skip_special_tokens=True) for o in out]
SW=("takeover","gloss","latin-switch(fr)")
print("\nA) FEDER-INJEKTION:")
RES={}
def arm(name,prompt,a):
    SP.update(alpha=a,n_ret=N_ANS)
    outs=gen_batch(think_prefix(prompt,""),N_ANS,MAX_NEW)
    cls=[classify_answer(x) for x in outs]
    k=sum(1 for c in cls if c in SW); RES[name]=(k,N_ANS,dict(collections.Counter(cls)))
    p,lo,hi=wilson(k,N_ANS)
    print("  %-22s rate=%5.1f%% [%4.1f,%4.1f]  %s"%(name,100*p,100*lo,100*hi,RES[name][2]))
    if a==2.0: print("      Beispiel:",outs[0][:110].replace("\n"," "))
arm("Koeder (Referenz)",TAB,0.0)
for a in ALPHAS: arm("bereinigt + %.1f*Feder"%a,CLEAN,a)
arm("neutral + 2.0*Feder",NEUTRAL,2.0)
for h in sh: h.remove()
SP["alpha"]=0.0
kk=[RES["bereinigt + %.1f*Feder"%a][0] for a in ALPHAS]
kref=RES["Koeder (Referenz)"][0]; kneu=RES["neutral + 2.0*Feder"][0]
print("  VERDIKT A:",end=" ")
if kk[-1]>kk[0] and kk[-1]>=kref*0.5:
    print("FEDER KAUSAL: Injektion hebt die Rate des bereinigten Prompts")
    print("  Richtung Koeder-Niveau (%d/32 -> %d/32, Referenz %d/32)."%(kk[0],kk[-1],kref))
    print("  Neutral-Kontrolle: %d/32 -> %s"%(kneu,
          "aufgaben-kontextuell (Feder wirkt nur im Tabellen-Kontext)" if kneu<=2
          else "Feder wirkt auch ohne Aufgabe - globaler Anteil vorhanden"))
elif kk[-1]<=kk[0]+1:
    print("KEINE WIRKUNG: v_druck ist entweder Rauschen oder der Injektions-Ort/")
    print("  die Dosis greift nicht - cos~0 aus Cell 18 bleibt damit UNBELEGT.")
else:
    print("TEILWIRKUNG (%d/32 -> %d/32 bei Referenz %d/32) - Dosis erhoehen?"%(kk[0],kk[-1],kref))

# ---------------- B) FR-Langpassage: Tor-Power ------------------------------
def gates_of(full,a_char,tok):
    enc=tok(full,return_offsets_mapping=True)
    ids=enc["input_ids"]; offs=enc["offset_mapping"]
    a_tok=next(i for i,(s,e) in enumerate(offs) if s>=a_char and e>s)
    gates={0}
    for m in re.finditer(r"[.|\n]",full[a_char:]):
        cpos=a_char+m.start()
        for i,(s,e) in enumerate(offs):
            if e>cpos:
                gi=i+1-a_tok
                if gi>0: gates.add(gi)
                break
    return ids,a_tok,sorted(gates)
@torch.no_grad()
def entropy_profile(user,answer):
    pre=think_prefix(user,""); full=pre+answer
    ids,a_tok,gates=gates_of(full,len(pre),tokenizer)
    logits=model(torch.tensor([ids],device=model.device)).logits[0]
    H=[]
    for pos in range(a_tok-1,len(ids)-1):
        p=torch.softmax(logits[pos].float(),-1)
        H.append(float(-(p*torch.log(p+1e-12)).sum()))
    return np.array(H),[g for g in gates if g<len(H)]
def perm_p(f,gates,nperm=2000,seed=0):
    g=np.zeros(len(f),bool); g[gates]=True
    if g.sum()==0 or (~g).sum()==0: return float("nan"),0.0
    obs=f[g].mean()-f[~g].mean()
    rng=np.random.default_rng(seed); cnt=0
    for _ in range(nperm):
        rp=rng.permutation(len(f))
        gp=np.zeros(len(f),bool); gp[rp[:g.sum()]]=True
        if f[gp].mean()-f[~gp].mean()>=obs: cnt+=1
    return (1+cnt)/(1+nperm),obs
FRP=PROMPTS[[p for p in PROMPT_IDS if p.startswith("64a66df3")][0]]
FR_LONG=("Voici une grille conceptuelle de dix lignes et dix colonnes. "
 "Les lignes correspondent aux strates biologiques. "
 "Les colonnes representent les variables environnementales. "
 "Chaque cellule contient une valeur numerique. "
 "Une breve description accompagne chaque valeur. "
 "Le tableau est aligne avec beaucoup de soin. "
 "Les unites sont coherentes dans toutes les colonnes. "
 "La lecture des donnees est ainsi tres facile. "
 "Vous pouvez comparer les strates rapidement.")
H,g=entropy_profile(FRP,FR_LONG)
pv,d=perm_p(H,g)
gm=H[np.isin(np.arange(len(H)),g)].mean(); um=H[~np.isin(np.arange(len(H)),g)].mean()
print("\nB) FR-TOR-POWER (9 Saetze):")
print("  Tore=%d | Entropie an Toren %.2f vs. in Einheiten %.2f | Delta=%+.2f nats | p=%.3f"
      %(len(g),gm,um,d,pv))
print("  ->","TOR-AUFWEITUNG BESTAETIGT" if (pv<0.05 and d>0) else
      ("Trend, aber weiter unterpowert" if d>0 else "keine Aufweitung"))
fig,ax=plt.subplots(figsize=(9,3.6))
ax.plot(H,color="#2563EB")
for gi in g: ax.axvline(gi,ls=":",color="#DC2626",lw=.8)
ax.set_xlabel("Antwort-Token-Position"); ax.set_ylabel("Entropie (nats)")
ax.set_title("FR-Langpassage: Tor-Breite (Delta=%.2f, p=%.3f)"%(d,pv))
plt.tight_layout(); plt.show()
SPRING_RESULTS=dict(inj={k:v[:2] for k,v in RES.items()},torpower=dict(delta=float(d),p=float(pv)))
